# Version 2: Adding more input and output features like age, gender, ados scores, etc

In [1]:
# ========================
# Multi-Task ADOS Prediction: Severity, Age, Gender, Social Affect, RRB, Comparison Score
# ========================
import numpy as np
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ========================
# 1. Enhanced preprocessing
# ========================
def load_npz_sequence(folder_path):
    frame_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.npz')])
    sequence = []
    for f in frame_files:
        data = np.load(os.path.join(folder_path, f))['coordinates']
        if data.shape[0] > 0:
            person_data = data[0]
            sequence.append(person_data)
    if not sequence:
        return np.empty((0, 24, 2))
    return np.array(sequence)

def normalize_skeleton(sequence):
    """Improved normalization with stability"""
    if sequence.shape[0] == 0:
        return sequence
    
    # Center around hip midpoint
    center = (sequence[:, 8:9, :] + sequence[:, 11:12, :]) / 2
    sequence = sequence - center
    
    # Normalize by torso length for scale invariance
    torso_length = np.linalg.norm(
        sequence[:, 1, :] - (sequence[:, 8, :] + sequence[:, 11, :]) / 2,
        axis=1, keepdims=True
    )
    torso_length = np.maximum(torso_length, 1e-6)  # Avoid division by zero
    sequence = sequence / (torso_length[:, :, np.newaxis] + 1e-6)
    
    return sequence

def pad_sequence(seq, max_len=500):
    num_frames, num_joints, _ = seq.shape
    if num_frames >= max_len:
        return seq[:max_len]
    else:
        pad_len = max_len - num_frames
        padding = np.zeros((pad_len, num_joints, 2))
        return np.concatenate([seq, padding], axis=0)

def compute_enhanced_features(sequence):
    """
    Enhanced feature extraction with more behavioral signals
    """
    num_frames = sequence.shape[0]
    
    # 1. Flattened joint positions (48 features)
    flat_joints = sequence.reshape(num_frames, -1)
    
    # 2. Velocity (48 features)
    velocity = np.zeros_like(flat_joints)
    velocity[1:] = flat_joints[1:] - flat_joints[:-1]
    
    # 3. Acceleration (48 features)
    acceleration = np.zeros_like(flat_joints)
    acceleration[1:] = velocity[1:] - velocity[:-1]
    
    # 4. Key joint distances (behavioral indicators)
    # Hand-hand distance
    left_wrist = sequence[:, 7, :]
    right_wrist = sequence[:, 4, :]
    dist_hands = np.linalg.norm(left_wrist - right_wrist, axis=1, keepdims=True)
    
    # Elbow-elbow distance
    left_elbow = sequence[:, 6, :]
    right_elbow = sequence[:, 3, :]
    dist_elbows = np.linalg.norm(left_elbow - right_elbow, axis=1, keepdims=True)
    
    # Hand-to-body distances (self-stimulatory behaviors)
    neck = sequence[:, 1, :]
    dist_left_hand_body = np.linalg.norm(left_wrist - neck, axis=1, keepdims=True)
    dist_right_hand_body = np.linalg.norm(right_wrist - neck, axis=1, keepdims=True)
    
    # 5. Movement magnitude (overall activity level)
    movement_mag = np.linalg.norm(velocity.reshape(num_frames, -1), axis=1, keepdims=True)
    
    # 6. Symmetry features (left-right coordination)
    left_shoulder = sequence[:, 5, :]
    right_shoulder = sequence[:, 2, :]
    shoulder_symmetry = np.linalg.norm(
        (left_wrist - left_shoulder) - (right_wrist - right_shoulder),
        axis=1, keepdims=True
    )
    
    # Combine all features
    features = np.concatenate([
        flat_joints,           # 48
        velocity,              # 48
        acceleration,          # 48
        dist_hands,            # 1
        dist_elbows,           # 1
        dist_left_hand_body,   # 1
        dist_right_hand_body,  # 1
        movement_mag,          # 1
        shoulder_symmetry      # 1
    ], axis=1)
    
    return features


In [2]:
# ========================
# 2. Load dataset - Use age/gender as INPUT features, predict ADOS metrics
# ========================
root_folder = "/kaggle/input/2d-coordinates"
metadata_file = "/kaggle/input/ados-rating/ADOS_rating.xlsx"
max_len = 100

def convert_age_to_years(age_str):
    """
    Convert age from format like "5Y,10M" to decimal years
    Example: "5Y,10M" -> 5 + 10/12 = 5.833
             "8Y,0M" -> 8.0
             "3Y,6M" -> 3.5
    """
    try:
        # If already a number, return it
        if isinstance(age_str, (int, float)):
            return float(age_str)
        
        # Convert to string and clean
        age_str = str(age_str).strip()
        
        # Split by comma
        parts = age_str.split(',')
        
        years = 0
        months = 0
        
        for part in parts:
            part = part.strip().upper()
            if 'Y' in part:
                years = int(part.replace('Y', '').strip())
            elif 'M' in part:
                months = int(part.replace('M', '').strip())
        
        # Convert to decimal years
        decimal_age = years + (months / 12.0)
        return decimal_age
    
    except Exception as e:
        print(f"Warning: Could not convert age '{age_str}': {e}")
        return None


# Load metadata
df = pd.read_excel(metadata_file)

# Display available columns for verification
print("Available columns in ADOS rating file:")
print(df.columns.tolist())
print(f"\nDataset has {len(df)} patients")

# Map patient IDs to data
# Targets (to predict): Severity (clasification), Social Affect, RRB, Comparison Score, ADOS Classification
# Attributes (input features): Age, Gender
target_columns = {
    'severity': 'Severity of Autism  ',
    'age': 'Chronological Age',
    'gender': 'Gender',
    'social_affect': 'Social Affect Total ',
    'rrb': 'Restricted and Repetitive Behavior (RRB) Total',
    'comparison_score': 'ADOS Comparison Score (1-10) <5  not very autistic. ASD people usually fall 5-10. 8-10=Severe, 5-7=moderate, 1-4=mild',
    'overall_total': 'Overall Total '
}

# Create mappings
id_to_data = {}
for idx, row in df.iterrows():
    patient_id = row['ID#']
    age_value = convert_age_to_years(row[target_columns['age']])
    # Skip if age conversion failed
    if age_value is None:
        print("Skipped")
        continue
    id_to_data[patient_id] = {
        'severity': row[target_columns['severity']],
        'age': age_value,
        'gender': row[target_columns['gender']],
        'social_affect': row[target_columns['social_affect']],
        'rrb': row[target_columns['rrb']],
        'comparison_score': row[target_columns['comparison_score']],
        'overall_total': row[target_columns['overall_total']]
    }

# Encode gender (categorical variable) - will be used as input feature
gender_encoder = LabelEncoder()
all_genders = [data['gender'] for data in id_to_data.values() if pd.notna(data['gender'])]
gender_encoder.fit(all_genders)
print(f"\nGender encoding (as input feature): {dict(zip(gender_encoder.classes_, gender_encoder.transform(gender_encoder.classes_)))}")

# Load sequences and labels
sequence_features = []  # Skeletal features
demographic_features = []  # Age and gender
labels = []  # Only ADOS targets
sequence_lengths = []
valid_patient_ids = []

for action in os.listdir(root_folder):
    print(f"Loading Action: {action}")
    action_path = os.path.join(root_folder, action)
    if not os.path.isdir(action_path):
        continue
    for trial in os.listdir(action_path):
        trial_path = os.path.join(action_path, trial)
        if not os.path.isdir(trial_path):
            continue
        try:
            patient_id = int(trial.split('_')[1])
        except (IndexError, ValueError):
            continue
        if patient_id not in id_to_data:
            continue
        
        sequence = load_npz_sequence(trial_path)
        if sequence.shape[0] == 0:
            continue
        
        # Get data for this patient
        patient_data = id_to_data[patient_id]
        
        # Check for missing values
        if any(pd.isna(patient_data[k]) for k in patient_data.keys()):
            continue
        
        # Encode gender for use as input
        gender_encoded = gender_encoder.transform([patient_data['gender']])[0]
        age_value = patient_data['age']
        
        # Process sequence
        original_length = min(sequence.shape[0], max_len)
        sequence = normalize_skeleton(sequence)
        sequence = pad_sequence(sequence, max_len)
        features = compute_enhanced_features(sequence)
        
        # Store skeletal features
        sequence_features.append(features)
        
        # Store demographic features (age, gender) - will be concatenated to model input
        demographic_features.append([age_value, gender_encoded])
        
        # Create label vector: [severity, social_affect, rrb, comparison_score, ados classification]
        label_vector = [
            patient_data['severity'],
            patient_data['social_affect'],
            patient_data['rrb'],
            patient_data['comparison_score']
        ]
        labels.append(label_vector)
        sequence_lengths.append(original_length)
        valid_patient_ids.append(patient_id)

sequence_features = np.array(sequence_features)
demographic_features = np.array(demographic_features)
labels = np.array(labels)
sequence_lengths = np.array(sequence_lengths)

print(f"\nSequence features shape: {sequence_features.shape}")
print(f"Demographic features shape: {demographic_features.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Number of samples: {len(sequence_features)}")

# Print statistics
print("\n" + "="*70)
print("INPUT FEATURES (Demographics)")
print("="*70)

print(f"Age            - Mean: {demographic_features[:, 0].mean():6.2f}, Std: {demographic_features[:, 0].std():6.2f}")

print(f"Gender (0/1)   - Mean: {demographic_features[:, 1].mean():6.2f}, Distribution: {np.bincount(demographic_features[:, 1].astype(int))}")

Available columns in ADOS rating file:
['ID#', 'Gender', 'Chronological Age', 'Module', 'Social Affect Total ', 'Restricted and Repetitive Behavior (RRB) Total', 'Overall Total ', 'ADOS-2 classification/Dx', 'ADOS Comparison Score (1-10) <5  not very autistic. ASD people usually fall 5-10. 8-10=Severe, 5-7=moderate, 1-4=mild', 'Severity of Autism  ']

Dataset has 41 patients

Gender encoding (as input feature): {'F': 0, 'M': 1}
Loading Action: arm_swing_as
Loading Action: maracas_forward_shaking_mfs
Loading Action: maracas_shaking_ms
Loading Action: tree_pose_tr
Loading Action: body_swing_bs
Loading Action: sing_and_clap_sac
Loading Action: frog_pose_fg
Loading Action: drumming_dr
Loading Action: squat_sq
Loading Action: twist_pose_tw
Loading Action: chest_expansion_ce

Sequence features shape: (1268, 100, 150)
Demographic features shape: (1268, 2)
Labels shape: (1268, 4)
Number of samples: 1268

INPUT FEATURES (Demographics)
Age            - Mean:   8.08, Std:   2.46
Gender (0/1)   - 

In [3]:
# Convert labels to numeric safely
labels = np.array(labels, dtype=object).astype(str)

clean_labels = []
for row in labels:
    clean_row = []
    for value in row:
        try:
            clean_value = float(value)
        except:
            # If string like '7*' or missing values → skip sample
            clean_value = None
        clean_row.append(clean_value)
    clean_labels.append(clean_row)

clean_labels = np.array(clean_labels, dtype=object)

# Remove rows with None
valid_mask = ~np.any(pd.isna(clean_labels).astype(bool), axis=1)
clean_labels = clean_labels[valid_mask]
sequence_features = sequence_features[valid_mask]
demographic_features = demographic_features[valid_mask]
sequence_lengths = sequence_lengths[valid_mask]

# Convert to float
labels = clean_labels.astype(float)
labels = np.array(labels)

In [4]:
# ========================
# 3. Feature normalization and target scaling
# ========================
# Normalize skeletal features
original_shape = sequence_features.shape
sequence_reshaped = sequence_features.reshape(-1, sequence_features.shape[-1])

feature_scaler = StandardScaler()
sequence_normalized = feature_scaler.fit_transform(sequence_reshaped)
sequence_features = sequence_normalized.reshape(original_shape)

# Normalize demographic features (age, gender)
demographic_scaler = StandardScaler()
demographic_features_normalized = demographic_scaler.fit_transform(demographic_features)

print("Feature scaling:")
print(f"  Skeletal features: StandardScaler applied")
print(f"  Age: StandardScaler applied (as input feature)")
print(f"  Gender: StandardScaler applied (as input feature)")

# Prepare targets (MIXED: 2 classification + 2 regression)
# Target 0: Severity - CLASSIFICATION (2 classes)
# Target 1: Social Affect - REGRESSION
# Target 2: RRB - REGRESSION  
# Target 3: Comparison Score - CLASSIFICATION (10 classes)

# For severity: Convert values to class indices
# Severity values are 2 or 3, map to classes 0 and 1
severity_values = labels[:, 0]
severity_unique = np.unique(severity_values)
print(f"\nSeverity unique values: {severity_unique}")
severity_mapping = {val: idx for idx, val in enumerate(sorted(severity_unique))}
print(f"Severity mapping: {severity_mapping}")
severity_classes = np.array([severity_mapping[val] for val in severity_values])

# For comparison score: Convert to class indices (1-10 -> 0-9)
comparison_values = labels[:, 3]
comparison_unique = np.unique(comparison_values)
print(f"\nComparison score unique values: {comparison_unique}")
comparison_classes = (comparison_values - 1).astype(int)  # 1-10 -> 0-9

# For social affect and RRB: Keep as regression with scaling
social_affect_scaler = StandardScaler()
rrb_scaler = StandardScaler()

social_affect_scaled = social_affect_scaler.fit_transform(labels[:, 1].reshape(-1, 1)).flatten()
rrb_scaled = rrb_scaler.fit_transform(labels[:, 2].reshape(-1, 1)).flatten()

# Store scalers for later use
target_scalers = [None, social_affect_scaler, rrb_scaler, None]  # None for classification tasks

print("\nTarget preparation:")
print(f"  Severity: Classification ({len(severity_unique)} classes) - {severity_unique}")
print(f"  Social Affect: Regression (Scaled)")
print(f"  RRB: Regression (Scaled)")
print(f"  Comparison Score: Classification (10 classes: 1-10)")

# Convert to torch tensors
X_sequence = torch.tensor(sequence_features, dtype=torch.float32)
X_demographic = torch.tensor(demographic_features_normalized, dtype=torch.float32)

# Create separate tensors for different target types
y_severity_class = torch.tensor(severity_classes, dtype=torch.long)
y_social_affect_reg = torch.tensor(social_affect_scaled, dtype=torch.float32)
y_rrb_reg = torch.tensor(rrb_scaled, dtype=torch.float32)
y_comparison_class = torch.tensor(comparison_classes, dtype=torch.long)

seq_lens = torch.tensor(sequence_lengths, dtype=torch.long)

# Store original labels for evaluation
y_original = torch.tensor(labels, dtype=torch.float32)

# Train/validation split
# Create dataset with all components
dataset = TensorDataset(
    X_sequence, X_demographic, 
    y_severity_class, y_social_affect_reg, y_rrb_reg, y_comparison_class,
    y_original, seq_lens
)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"\nDataset split:")
print(f"  Training samples: {train_size}")
print(f"  Validation samples: {val_size}")
print(f"  Batch size: 32")
# Store severity mapping for inverse transform
severity_inverse_mapping = {idx: val for val, idx in severity_mapping.items()}

# ========================
# HANDLE CLASS IMBALANCE - Calculate class weights
# ========================
from sklearn.utils.class_weight import compute_class_weight

# Calculate weights for severity classes
# Severity has 2 classes, ensure we have weights for both
severity_classes_present = np.unique(severity_classes)
severity_weights_computed = compute_class_weight(
    'balanced',
    classes=severity_classes_present,
    y=severity_classes
)

# Create full weight tensor for all severity classes (2 classes)
num_severity_classes = len(severity_inverse_mapping)
severity_class_weights = torch.ones(num_severity_classes, dtype=torch.float32)
for idx, class_idx in enumerate(severity_classes_present):
    severity_class_weights[class_idx] = severity_weights_computed[idx]

# Calculate weights for comparison score classes
# Comparison has 10 classes (0-9 representing scores 1-10)
comparison_classes_present = np.unique(comparison_classes)
comparison_weights_computed = compute_class_weight(
    'balanced',
    classes=comparison_classes_present,
    y=comparison_classes
)

# Create full weight tensor for all 10 comparison classes
comparison_class_weights = torch.ones(10, dtype=torch.float32)
for idx, class_idx in enumerate(comparison_classes_present):
    comparison_class_weights[class_idx] = comparison_weights_computed[idx]

for class_idx in range(10):
    if class_idx not in comparison_classes_present:
        comparison_class_weights[class_idx] = 0.0

print("\n" + "="*70)
print("CLASS IMBALANCE HANDLING")
print("="*70)
print(f"Severity class distribution: {np.bincount(severity_classes)}")
print(f"Severity classes present: {severity_classes_present}")
print(f"Severity class weights (all {num_severity_classes} classes): {severity_class_weights.numpy()}")
print(f"\nComparison score class distribution: {np.bincount(comparison_classes)}")
print(f"Comparison classes present: {comparison_classes_present}")
print(f"Comparison class weights (all 10 classes): {comparison_class_weights.numpy()}")
print("="*70)


Feature scaling:
  Skeletal features: StandardScaler applied
  Age: StandardScaler applied (as input feature)
  Gender: StandardScaler applied (as input feature)

Severity unique values: [2. 3.]
Severity mapping: {2.0: 0, 3.0: 1}

Comparison score unique values: [ 6.  7.  8.  9. 10.]

Target preparation:
  Severity: Classification (2 classes) - [2. 3.]
  Social Affect: Regression (Scaled)
  RRB: Regression (Scaled)
  Comparison Score: Classification (10 classes: 1-10)

Dataset split:
  Training samples: 1014
  Validation samples: 254
  Batch size: 32

CLASS IMBALANCE HANDLING
Severity class distribution: [ 241 1027]
Severity classes present: [0 1]
Severity class weights (all 2 classes): [2.6307054  0.61733204]

Comparison score class distribution: [  0   0   0   0   0  53 188 286  88 653]
Comparison classes present: [5 6 7 8 9]
Comparison class weights (all 10 classes): [0.         0.         0.         0.         0.         4.7849054
 1.3489362  0.88671327 2.8818183  0.3883614 ]


In [5]:
# ========================
# ADVERSARIAL TRAINING FUNCTIONS FOR MODEL ROBUSTNESS
# ========================
import torch
import torch.nn as nn

def fgsm_attack(model, x_seq, x_demo, seq_len, severity_target, social_affect_target, 
                rrb_target, comparison_target, severity_weights=None, comparison_weights=None, epsilon=0.01):
    """
    Fast Gradient Sign Method (FGSM) attack
    Generates adversarial examples by adding small perturbations in the direction of gradient
    
    Args:
        model: The neural network model
        x_seq: Input sequence [batch, seq_len, features]
        x_demo: Demographic features [batch, demo_features]
        seq_len: Sequence lengths [batch]
        targets: Ground truth targets for all tasks
        severity_weights: Class weights for severity (handles imbalance)
        comparison_weights: Class weights for comparison score (handles imbalance)
        epsilon: Perturbation magnitude (default: 0.01)
    
    Returns:
        x_seq_adv: Adversarial sequence with perturbations
    """
    model.eval()  # Set to eval mode to get gradients
    
    # Clone and require gradients
    x_seq_adv = x_seq.clone().detach().requires_grad_(True)
    x_demo_adv = x_demo.clone().detach()
    
    # Forward pass
    outputs = model(x_seq_adv, x_demo_adv, seq_len)
    
    # Compute loss
    loss, _ = multi_task_loss(outputs, severity_target, social_affect_target, 
                              rrb_target, comparison_target,
                          severity_class_weights=severity_weights,
                          comparison_class_weights=comparison_weights)
    
    # Backward pass to get gradients
    model.zero_grad()
    loss.backward()
    
    # Create adversarial example using sign of gradients
    perturbation = epsilon * x_seq_adv.grad.sign()
    x_seq_adv = x_seq.detach() + perturbation
    
    # Clip to maintain valid range (assuming normalized data)
    x_seq_adv = torch.clamp(x_seq_adv, -5, 5)
    
    return x_seq_adv.detach()


def pgd_attack(model, x_seq, x_demo, seq_len, severity_target, social_affect_target,
               rrb_target, comparison_target, epsilon=0.01, alpha=0.002,  severity_weights=None, comparison_weights=None, num_iter=7):
    """
    Projected Gradient Descent (PGD) attack
    More powerful iterative attack that applies FGSM multiple times
    
    Args:
        model: The neural network model
        x_seq: Input sequence
        x_demo: Demographic features
        seq_len: Sequence lengths
        targets: Ground truth targets for all tasks
        epsilon: Maximum perturbation magnitude
        severity_weights: Class weights for severity (handles imbalance)
        comparison_weights: Class weights for comparison score (handles imbalance)
        alpha: Step size per iteration
        num_iter: Number of iterations
    
    Returns:
        x_seq_adv: Adversarial sequence
    """
    model.eval()
    
    x_seq_adv = x_seq.clone().detach()
    x_demo_adv = x_demo.clone().detach()
    
    # Random initialization within epsilon ball
    x_seq_adv = x_seq_adv + torch.empty_like(x_seq_adv).uniform_(-epsilon, epsilon)
    x_seq_adv = torch.clamp(x_seq_adv, -5, 5)
    
    for i in range(num_iter):
        x_seq_adv.requires_grad = True
        
        outputs = model(x_seq_adv, x_demo_adv, seq_len)
        loss, _ = multi_task_loss(outputs, severity_target, social_affect_target,
                                  rrb_target, comparison_target,
                          severity_class_weights=severity_class_weights,
                          comparison_class_weights=comparison_weights)
        
        model.zero_grad()
        loss.backward()
        
        # Take step in direction of gradient
        with torch.no_grad():
            perturbation = alpha * x_seq_adv.grad.sign()
            x_seq_adv = x_seq_adv.detach() + perturbation
            
            # Project back to epsilon ball around original input
            delta = torch.clamp(x_seq_adv - x_seq, min=-epsilon, max=epsilon)
            x_seq_adv = x_seq + delta
            x_seq_adv = torch.clamp(x_seq_adv, -5, 5)
    
    return x_seq_adv.detach()


def robust_data_augmentation(x_seq, augmentation_type='noise', noise_level=0.01):
    """
    Apply robust data augmentation to improve model generalization
    
    Args:
        x_seq: Input sequence [batch, seq_len, features]
        augmentation_type: Type of augmentation ('noise', 'smooth', 'spatial')
        noise_level: Magnitude of augmentation
    
    Returns:
        x_seq_aug: Augmented sequence
    """
    x_seq_aug = x_seq.clone()
    
    if augmentation_type == 'noise':
        # Add Gaussian noise
        noise = torch.randn_like(x_seq) * noise_level
        x_seq_aug = x_seq + noise
        
    elif augmentation_type == 'smooth':
        # Temporal smoothing using moving average
        kernel_size = 3
        padding = kernel_size // 2
        x_seq_aug = torch.nn.functional.avg_pool1d(
            x_seq.transpose(1, 2), 
            kernel_size=kernel_size, 
            stride=1, 
            padding=padding
        ).transpose(1, 2)
        
    elif augmentation_type == 'spatial':
        # Small spatial perturbations (simulating keypoint detection noise)
        spatial_noise = torch.randn_like(x_seq) * (noise_level * 0.5)
        x_seq_aug = x_seq + spatial_noise
    
    # Clip to valid range
    x_seq_aug = torch.clamp(x_seq_aug, -5, 5)
    
    return x_seq_aug


def adversarial_training_step(model, optimizer, x_seq, x_demo, seq_len, 
                               severity_target, social_affect_target, 
                               rrb_target, comparison_target, severity_weights=None, comparison_weights=None,
                               adv_ratio=0.3, attack_type='fgsm'):
    """
    Perform one training step with adversarial examples mixed in
    
    Args:
        model: Neural network model
        optimizer: Optimizer
        inputs: All inputs and targets
        adv_ratio: Ratio of adversarial examples (0.0-1.0)
        severity_weights: Class weights for severity (handles imbalance)
        comparison_weights: Class weights for comparison score (handles imbalance)
        attack_type: 'fgsm' or 'pgd'
    
    Returns:
        loss: Training loss
        losses_dict: Individual task losses
    """
    batch_size = x_seq.size(0)
    adv_size = int(batch_size * adv_ratio)
    
    if adv_size > 0:
        # Generate adversarial examples for part of the batch
        if attack_type == 'fgsm':
            x_seq_adv = fgsm_attack(
                model, x_seq[:adv_size], x_demo[:adv_size], seq_len[:adv_size],
                severity_target[:adv_size], social_affect_target[:adv_size],
                rrb_target[:adv_size], comparison_target[:adv_size],
                severity_weights=severity_class_weights,
                comparison_weights=comparison_class_weights,
                epsilon=0.01
            )
        elif attack_type == 'pgd':
            x_seq_adv = pgd_attack(
                model, x_seq[:adv_size], x_demo[:adv_size], seq_len[:adv_size],
                severity_target[:adv_size], social_affect_target[:adv_size],
                rrb_target[:adv_size], comparison_target[:adv_size],
                severity_weights=severity_class_weights,
                comparison_weights=comparison_class_weights,
                epsilon=0.01, alpha=0.002, num_iter=5
            )
        
        # Combine clean and adversarial examples
        x_seq_combined = torch.cat([x_seq_adv, x_seq[adv_size:]], dim=0)
    else:
        x_seq_combined = x_seq
    
    # Forward pass on combined batch
    model.train()
    optimizer.zero_grad()
    outputs = model(x_seq_combined, x_demo, seq_len)
    
    # Compute loss
    loss, losses_dict = multi_task_loss(
        outputs, severity_target, social_affect_target, 
        rrb_target, comparison_target,
                          severity_class_weights=severity_class_weights,
                          comparison_class_weights=comparison_class_weights
    )
    
    # Backward pass
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    return loss, losses_dict


print("✅ Adversarial training functions defined:")
print("   - fgsm_attack(): Fast Gradient Sign Method")
print("   - pgd_attack(): Projected Gradient Descent")
print("   - robust_data_augmentation(): Data augmentation for robustness")
print("   - adversarial_training_step(): Combined training step")
print("\n🛡️ These functions will make the model robust against adversarial attacks")

✅ Adversarial training functions defined:
   - fgsm_attack(): Fast Gradient Sign Method
   - pgd_attack(): Projected Gradient Descent
   - robust_data_augmentation(): Data augmentation for robustness
   - adversarial_training_step(): Combined training step

🛡️ These functions will make the model robust against adversarial attacks


In [ ]:
# ========================
# 4. Multi-Task ADOS Model with Demographic Input Features
# ========================
class ImprovedADOSModel(nn.Module):
    def __init__(self, sequence_input_size, demographic_input_size=2, 
                 hidden_size=128, num_layers=2, dropout=0.4, num_outputs=4):
        super(ImprovedADOSModel, self).__init__()
        
        self.num_outputs = num_outputs
        self.demographic_input_size = demographic_input_size
        
        # Bidirectional LSTM for sequence processing (shared for all tasks)
        self.lstm = nn.LSTM(
            sequence_input_size, hidden_size, num_layers,
            batch_first=True, bidirectional=True, 
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Attention mechanism (shared)
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )
        
        # Layer normalization
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        # Demographic feature processing
        self.demographic_fc = nn.Sequential(
            nn.Linear(demographic_input_size, 16),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5)
        )
        
        # Shared representation layer (combines sequence + demographic)
        combined_size = hidden_size * 2 + 16
        self.shared_fc = nn.Sequential(
            nn.Linear(combined_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Task-specific heads (all regression)
        # Output 0: Severity
        self.severity_head = nn.Linear(64, 2)
        
        # Output 1: Social Affect
        self.social_affect_head = nn.Linear(64, 1)
        
        # Output 2: RRB
        self.rrb_head = nn.Linear(64, 1)
        
        # Output 3: Comparison Score
        self.comparison_head = nn.Linear(64, 10)
    
    def forward(self, x_sequence, x_demographic, seq_lengths=None):
        # LSTM processing for sequence data
        lstm_out, _ = self.lstm(x_sequence)
        
        # Apply layer normalization
        lstm_out = self.layer_norm(lstm_out)
        
        # Attention mechanism
        attention_weights = self.attention(lstm_out)
        attention_weights = torch.softmax(attention_weights, dim=1)
        
        # Weighted sum of sequence features
        sequence_context = torch.sum(attention_weights * lstm_out, dim=1)
        
        # Process demographic features
        demographic_repr = self.demographic_fc(x_demographic)
        
        # Combine sequence and demographic representations
        combined = torch.cat([sequence_context, demographic_repr], dim=1)
        
        # Shared representation
        shared_repr = self.shared_fc(combined)
        
        severity_logits = self.severity_head(shared_repr)  # [batch, 2]
        social_affect = self.social_affect_head(shared_repr)  # [batch, 1]
        rrb = self.rrb_head(shared_repr)  # [batch, 1]
        comparison_logits = self.comparison_head(shared_repr)  # [batch, 10]
        
        # Return as dictionary for clarity
        return {
            'severity_logits': severity_logits,
            'social_affect': social_affect,
            'rrb': rrb,
            'comparison_logits': comparison_logits
        }
        
        # Concatenate outputs: [batch, 4]
        outputs = torch.cat([severity, social_affect, rrb, comparison], dim=1)
        
        return outputs

print("Multi-Task Model Architecture:")
print("  Input: Sequence features (150 per frame) + Demographic features (age, gender)")
print("  Shared: Bidirectional LSTM + Attention + Demographic FC + Combined FC")
print("  Task-specific heads (all regression):")
print("    - Severity")
print("    - Social Affect")
print("    - RRB")
print("    - Comparison Score")
print("  Demographic features used as AUXILIARY INPUT to improve predictions")

In [7]:
# ========================
# 5. Multi-Task Training with Weighted Loss (4 ADOS Metrics)
# ========================
sequence_input_size = sequence_features.shape[2]
demographic_input_size = demographic_features.shape[1]

model = ImprovedADOSModel(
    sequence_input_size=sequence_input_size,
    demographic_input_size=demographic_input_size,
    hidden_size=64,
    num_layers=1,
    dropout=0.3,
    num_outputs=4
)
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = nn.CrossEntropyLoss(weight=self.weight, reduction='none')(logits, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# Multi-task loss function (2 classification + 2 regression)
def multi_task_loss(predictions, severity_target, social_affect_target, rrb_target, comparison_target, task_weights=None, severity_class_weights=None, comparison_class_weights=None):
    """
    Compute weighted multi-task loss for mixed classification and regression
    
    predictions: dict with keys 'severity_logits', 'social_affect', 'rrb', 'comparison_logits'
    severity_target: [batch] - class labels (0 or 1)
    social_affect_target: [batch] - scaled regression values
    rrb_target: [batch] - scaled regression values
    comparison_target: [batch] - class labels (0-9)
    """
    if task_weights is None:
        task_weights = {
        'severity': 2.0,       # was 1.5
        'social_affect': 1.0,  # was 1.3
        'rrb': 1.0,            # was 1.3
        'comparison': 2.5      # was 1.0 — this was the most collapsed
    }
    
    # Classification losses
    loss_severity = FocalLoss(weight=severity_class_weights, gamma=2.0)(
        predictions['severity_logits'], severity_target
    )
    loss_comparison = FocalLoss(weight=comparison_class_weights, gamma=2.0)(
        predictions['comparison_logits'], comparison_target
    )
    
    # Regression losses
    huber = nn.HuberLoss(delta=1.0)
    loss_social_affect = huber(predictions['social_affect'].squeeze(), social_affect_target)
    loss_rrb = huber(predictions['rrb'].squeeze(), rrb_target)
    
    # Weighted total loss
    total_loss = (
        task_weights['severity'] * loss_severity +
        task_weights['social_affect'] * loss_social_affect +
        task_weights['rrb'] * loss_rrb +
        task_weights['comparison'] * loss_comparison
    )
    
    # Return total loss and individual losses for monitoring
    losses_dict = {
        'total': total_loss,
        'severity': loss_severity,
        'social_affect': loss_social_affect,
        'rrb': loss_rrb,
        'comparison': loss_comparison
    }
    
    return total_loss, losses_dict

# Optimizer with weight decay
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

# Early stopping
best_val_loss = float('inf')
patience = 15
patience_counter = 0

num_epochs = 100
# 🛡️ ADVERSARIAL TRAINING CONFIGURATION
USE_ADVERSARIAL_TRAINING = True  # Set to False to disable adversarial training
ADVERSARIAL_RATIO = 0.3  # 30% of each batch will be adversarial examples
ATTACK_TYPE = 'fgsm'  # 'fgsm' or 'pgd'

print("\n" + "="*70)
print("STARTING MULTI-TASK TRAINING (4 ADOS METRICS)")
print("="*70)
print(f"Epochs: {num_epochs}")
print(f"Learning rate: 1e-3")
print(f"Early stopping patience: {patience}")
print(f"🛡️ Adversarial Training: {'ENABLED' if USE_ADVERSARIAL_TRAINING else 'DISABLED'}")
if USE_ADVERSARIAL_TRAINING:
    print(f"   Attack Type: {ATTACK_TYPE.upper()}")
    print(f"   Adversarial Ratio: {ADVERSARIAL_RATIO*100:.0f}%")
print(f"Using Age and Gender as INPUT features to improve predictions")
print("="*70 + "\n")

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0
    train_losses_sum = {k: 0.0 for k in ['severity', 'social_affect', 'rrb', 'comparison']}
    
    for batch in train_loader:
        x_seq, x_demo, y_sev_class, y_sa_reg, y_rrb_reg, y_comp_class, y_orig, lens = batch
        if USE_ADVERSARIAL_TRAINING:
            # Use adversarial training step
            loss, losses_dict = adversarial_training_step(
                model, optimizer, x_seq, x_demo, lens,
                y_sev_class, y_sa_reg, y_rrb_reg, y_comp_class,
                adv_ratio=ADVERSARIAL_RATIO, attack_type=ATTACK_TYPE,
                          severity_weights=severity_class_weights,
                          comparison_weights=comparison_class_weights
            )
        else:
            # Standard training step
            optimizer.zero_grad()
            out = model(x_seq, x_demo, lens)
            loss, losses_dict = multi_task_loss(out, y_sev_class, y_sa_reg, y_rrb_reg, y_comp_class,
                          severity_class_weights=severity_weights,
                          comparison_class_weights=comparison_class_weights)
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
        
        train_loss += loss.item() * x_seq.size(0)
        for k in train_losses_sum.keys():
            train_losses_sum[k] += losses_dict[k].item() * x_seq.size(0)
    
    train_loss /= len(train_loader.dataset)
    for k in train_losses_sum.keys():
        train_losses_sum[k] /= len(train_loader.dataset)
    
    # Validation
    model.eval()
    val_loss = 0
    val_losses_sum = {k: 0.0 for k in ['severity', 'social_affect', 'rrb', 'comparison']}
    
    for batch in val_loader:
            x_seq, x_demo, y_sev_class, y_sa_reg, y_rrb_reg, y_comp_class, y_orig, lens = batch
            
            out = model(x_seq, x_demo, lens)
            loss, losses_dict = multi_task_loss(out, y_sev_class, y_sa_reg, y_rrb_reg, y_comp_class)
            
            val_loss += loss.item() * x_seq.size(0)
            for k in val_losses_sum.keys():
                val_losses_sum[k] += losses_dict[k].item() * x_seq.size(0)
    
    val_loss /= len(val_loader.dataset)
    for k in val_losses_sum.keys():
        val_losses_sum[k] /= len(val_loader.dataset)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"  Task Losses (Val):")
    print(f"    Severity (CE): {val_losses_sum['severity']:.4f}")
    print(f"    Social Affect (Huber): {val_losses_sum['social_affect']:.4f}")
    print(f"    RRB (Huber): {val_losses_sum['rrb']:.4f}")
    print(f"    Comparison (CE): {val_losses_sum['comparison']:.4f}")
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_ados_multitask_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

# Load best model
model.load_state_dict(torch.load('best_ados_multitask_model.pth'))
print("\n✅ Best model loaded")

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



STARTING MULTI-TASK TRAINING (4 ADOS METRICS)
Epochs: 100
Learning rate: 1e-3
Early stopping patience: 15
🛡️ Adversarial Training: ENABLED
   Attack Type: FGSM
   Adversarial Ratio: 30%
Using Age and Gender as INPUT features to improve predictions

Epoch 1/100
  Train Loss: 5.0229 | Val Loss: 4.2845
  Task Losses (Val):
    Severity (CE): 0.2670
    Social Affect (Huber): 0.4472
    RRB (Huber): 0.2755
    Comparison (CE): 1.2111
Epoch 2/100
  Train Loss: 3.8047 | Val Loss: 3.6870
  Task Losses (Val):
    Severity (CE): 0.2347
    Social Affect (Huber): 0.4055
    RRB (Huber): 0.2567
    Comparison (CE): 1.0222
Epoch 3/100
  Train Loss: 2.9743 | Val Loss: 3.1499
  Task Losses (Val):
    Severity (CE): 0.1929
    Social Affect (Huber): 0.3733
    RRB (Huber): 0.2469
    Comparison (CE): 0.8576
Epoch 4/100
  Train Loss: 2.2120 | Val Loss: 2.3477
  Task Losses (Val):
    Severity (CE): 0.0954
    Social Affect (Huber): 0.3387
    RRB (Huber): 0.2274
    Comparison (CE): 0.6363
Epoch 5/10

In [9]:
# ========================
# 6. Comprehensive Evaluation Metrics (UPDATED FOR MIXED TASKS)
# ========================
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

model.eval()

# Collect predictions and targets
all_severity_preds = []
all_severity_targets = []
all_social_affect_preds = []
all_social_affect_targets = []
all_rrb_preds = []
all_rrb_targets = []
all_comparison_preds = []
all_comparison_targets = []
all_original_labels = []

with torch.no_grad():
    for batch in val_loader:
        x_seq, x_demo, y_sev_class, y_sa_reg, y_rrb_reg, y_comp_class, y_orig, lens = batch
        
        out = model(x_seq, x_demo, lens)
        
        # Get class predictions (argmax for classification)
        severity_pred_class = torch.argmax(out['severity_logits'], dim=1)
        comparison_pred_class = torch.argmax(out['comparison_logits'], dim=1)
        
        all_severity_preds.append(severity_pred_class.numpy())
        all_severity_targets.append(y_sev_class.numpy())
        
        all_social_affect_preds.append(out['social_affect'].squeeze().numpy())
        all_social_affect_targets.append(y_sa_reg.numpy())
        
        all_rrb_preds.append(out['rrb'].squeeze().numpy())
        all_rrb_targets.append(y_rrb_reg.numpy())
        
        all_comparison_preds.append(comparison_pred_class.numpy())
        all_comparison_targets.append(y_comp_class.numpy())
        
        all_original_labels.append(y_orig.numpy())

# Concatenate all batches
severity_pred_classes = np.concatenate(all_severity_preds)
severity_true_classes = np.concatenate(all_severity_targets)

social_affect_pred_scaled = np.concatenate(all_social_affect_preds)
social_affect_true_scaled = np.concatenate(all_social_affect_targets)

rrb_pred_scaled = np.concatenate(all_rrb_preds)
rrb_true_scaled = np.concatenate(all_rrb_targets)

comparison_pred_classes = np.concatenate(all_comparison_preds)
comparison_true_classes = np.concatenate(all_comparison_targets)

original_labels = np.concatenate(all_original_labels)

# Convert predictions back to original values
# Severity: map class indices back to original values (2 or 3)
severity_pred_original = np.array([severity_inverse_mapping[cls] for cls in severity_pred_classes])
severity_true_original = original_labels[:, 0]

# Social Affect and RRB: inverse transform
social_affect_pred_original = social_affect_scaler.inverse_transform(social_affect_pred_scaled.reshape(-1, 1)).flatten()
social_affect_true_original = original_labels[:, 1]

rrb_pred_original = rrb_scaler.inverse_transform(rrb_pred_scaled.reshape(-1, 1)).flatten()
rrb_true_original = original_labels[:, 2]

# Comparison Score: add 1 to convert from 0-9 to 1-10
comparison_pred_original = comparison_pred_classes + 1
comparison_true_original = original_labels[:, 3].astype(int)

print("\n" + "="*80)
print("FINAL VALIDATION METRICS - MIXED CLASSIFICATION & REGRESSION")
print("="*80)
print("Using Age and Gender as INPUT features")
print("="*80)

# ===== SEVERITY (CLASSIFICATION) =====
print("\n" + "-"*80)
print("SEVERITY - CLASSIFICATION (2 classes)")
print("-"*80)
acc_severity = accuracy_score(severity_true_classes, severity_pred_classes)
print(f"Accuracy: {acc_severity:.4f}")

# Precision, Recall, F1
prec, rec, f1, support = precision_recall_fscore_support(severity_true_classes, severity_pred_classes, average='weighted')
print(f"Precision (weighted): {prec:.4f}")
print(f"Recall (weighted): {rec:.4f}")
print(f"F1-Score (weighted): {f1:.4f}")

# Confusion Matrix
cm_severity = confusion_matrix(severity_true_classes, severity_pred_classes)
print("\nConfusion Matrix:")
print(cm_severity)

# Classification Report
print("\nDetailed Classification Report:")
severity_class_names = [f"Severity {severity_inverse_mapping[i]}" for i in range(len(severity_inverse_mapping))]
print(classification_report(severity_true_classes, severity_pred_classes, target_names=severity_class_names))

# ===== SOCIAL AFFECT (REGRESSION) =====
print("\n" + "-"*80)
print("SOCIAL AFFECT - REGRESSION")
print("-"*80)
mae_sa = mean_absolute_error(social_affect_true_original, social_affect_pred_original)
rmse_sa = np.sqrt(mean_squared_error(social_affect_true_original, social_affect_pred_original))
r2_sa = r2_score(social_affect_true_original, social_affect_pred_original)
relative_mae_sa = (mae_sa / np.mean(social_affect_true_original)) * 100 if np.mean(social_affect_true_original) != 0 else 0

print(f"MAE:  {mae_sa:.4f}")
print(f"RMSE: {rmse_sa:.4f}")
print(f"R²:   {r2_sa:.4f}")
print(f"Relative MAE: {relative_mae_sa:.2f}%")
print(f"Mean True Value: {np.mean(social_affect_true_original):.2f}")

# ===== RRB (REGRESSION) =====
print("\n" + "-"*80)
print("RRB - REGRESSION")
print("-"*80)
mae_rrb = mean_absolute_error(rrb_true_original, rrb_pred_original)
rmse_rrb = np.sqrt(mean_squared_error(rrb_true_original, rrb_pred_original))
r2_rrb = r2_score(rrb_true_original, rrb_pred_original)
relative_mae_rrb = (mae_rrb / np.mean(rrb_true_original)) * 100 if np.mean(rrb_true_original) != 0 else 0

print(f"MAE:  {mae_rrb:.4f}")
print(f"RMSE: {rmse_rrb:.4f}")
print(f"R²:   {r2_rrb:.4f}")
print(f"Relative MAE: {relative_mae_rrb:.2f}%")
print(f"Mean True Value: {np.mean(rrb_true_original):.2f}")

# ===== COMPARISON SCORE (CLASSIFICATION) =====
print("\n" + "-"*80)
print("COMPARISON SCORE - CLASSIFICATION (10 classes: 1-10)")
print("-"*80)
acc_comparison = accuracy_score(comparison_true_classes, comparison_pred_classes)
print(f"Accuracy: {acc_comparison:.4f}")

# Precision, Recall, F1
prec_comp, rec_comp, f1_comp, support_comp = precision_recall_fscore_support(
    comparison_true_classes, comparison_pred_classes, average='weighted', zero_division=0
)
print(f"Precision (weighted): {prec_comp:.4f}")
print(f"Recall (weighted): {rec_comp:.4f}")
print(f"F1-Score (weighted): {f1_comp:.4f}")

# MAE for comparison (treating as ordinal)
mae_comparison = mean_absolute_error(comparison_true_original, comparison_pred_original)
print(f"MAE (ordinal): {mae_comparison:.4f}")

# Confusion Matrix
# Get unique classes that actually appear in the data
unique_comparison_classes = np.unique(np.concatenate([comparison_true_classes, comparison_pred_classes]))
cm_comparison = confusion_matrix(comparison_true_classes, comparison_pred_classes, labels=unique_comparison_classes)
print("\nConfusion Matrix:")
print(cm_comparison)

# Classification Report
print("\nDetailed Classification Report:")
# Only create names for classes that actually exist
comparison_class_names = [f"Score {i+1}" for i in unique_comparison_classes]
print(classification_report(comparison_true_classes, comparison_pred_classes, 
                          labels=unique_comparison_classes,
                          target_names=comparison_class_names, zero_division=0))

print("\n" + "="*80)
print("OVERALL PERFORMANCE SUMMARY:")
print("-" * 80)
print(f"Classification Tasks:")
print(f"  Severity - Accuracy: {acc_severity:.4f}, F1: {f1:.4f}")
print(f"  Comparison Score - Accuracy: {acc_comparison:.4f}, F1: {f1_comp:.4f}, MAE: {mae_comparison:.4f}")
print(f"\nRegression Tasks:")
print(f"  Social Affect - R²: {r2_sa:.4f}, MAE: {mae_sa:.4f}")
print(f"  RRB - R²: {r2_rrb:.4f}, MAE: {mae_rrb:.4f}")
print("="*80)

# Save predictions for further analysis
results_df = pd.DataFrame({
    'Severity_True': severity_true_original,
    'Severity_Pred': severity_pred_original,
    'SocialAffect_True': social_affect_true_original,
    'SocialAffect_Pred': social_affect_pred_original,
    'RRB_True': rrb_true_original,
    'RRB_Pred': rrb_pred_original,
    'ComparisonScore_True': comparison_true_original,
    'ComparisonScore_Pred': comparison_pred_original
})

results_df.to_csv('multitask_predictions.csv', index=False)
print("\n✅ Predictions saved to 'multitask_predictions.csv'")


FINAL VALIDATION METRICS - MIXED CLASSIFICATION & REGRESSION
Using Age and Gender as INPUT features

--------------------------------------------------------------------------------
SEVERITY - CLASSIFICATION (2 classes)
--------------------------------------------------------------------------------
Accuracy: 0.9843
Precision (weighted): 0.9846
Recall (weighted): 0.9843
F1-Score (weighted): 0.9843

Confusion Matrix:
[[ 56   1]
 [  3 194]]

Detailed Classification Report:
              precision    recall  f1-score   support

Severity 2.0       0.95      0.98      0.97        57
Severity 3.0       0.99      0.98      0.99       197

    accuracy                           0.98       254
   macro avg       0.97      0.98      0.98       254
weighted avg       0.98      0.98      0.98       254


--------------------------------------------------------------------------------
SOCIAL AFFECT - REGRESSION
--------------------------------------------------------------------------------
MAE:  

In [ ]:
# ========================
# 7. Visualization of Multi-Task Results (UPDATED FOR MIXED TASKS)
# ========================
import matplotlib.pyplot as plt
import seaborn as sns

fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

fig.suptitle('Multi-Task ADOS Prediction Results (Mixed Classification & Regression)', 
             fontsize=16, fontweight='bold')

# ===== SEVERITY (CLASSIFICATION - Confusion Matrix) =====
ax1 = fig.add_subplot(gs[0, 0])
sns.heatmap(cm_severity, annot=True, fmt='d', cmap='Blues', ax=ax1, cbar=True)
ax1.set_xlabel('Predicted Class')
ax1.set_ylabel('True Class')
ax1.set_title(f'Severity (Classification)\nAccuracy: {acc_severity:.3f}, F1: {f1:.3f}')
ax1.set_xticklabels([f'Sev {severity_inverse_mapping[i]}' for i in range(len(severity_inverse_mapping))])
ax1.set_yticklabels([f'Sev {severity_inverse_mapping[i]}' for i in range(len(severity_inverse_mapping))])

# ===== SOCIAL AFFECT (REGRESSION - Scatter Plot) =====
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(social_affect_true_original, social_affect_pred_original, alpha=0.6, s=50, color='steelblue')
min_val = min(social_affect_true_original.min(), social_affect_pred_original.min())
max_val = max(social_affect_true_original.max(), social_affect_pred_original.max())
ax2.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
ax2.set_xlabel('True Values', fontsize=11)
ax2.set_ylabel('Predicted Values', fontsize=11)
ax2.set_title(f'Social Affect (Regression)\nMAE: {mae_sa:.3f}, R²: {r2_sa:.3f}', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

# ===== RRB (REGRESSION - Scatter Plot) =====
ax3 = fig.add_subplot(gs[1, 0])
ax3.scatter(rrb_true_original, rrb_pred_original, alpha=0.6, s=50, color='steelblue')
min_val = min(rrb_true_original.min(), rrb_pred_original.min())
max_val = max(rrb_true_original.max(), rrb_pred_original.max())
ax3.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
ax3.set_xlabel('True Values', fontsize=11)
ax3.set_ylabel('Predicted Values', fontsize=11)
ax3.set_title(f'RRB (Regression)\nMAE: {mae_rrb:.3f}, R²: {r2_rrb:.3f}', fontsize=12)
ax3.legend()
ax3.grid(True, alpha=0.3)

# ===== COMPARISON SCORE (CLASSIFICATION - Confusion Matrix) =====
ax4 = fig.add_subplot(gs[1, 1])
# Show only non-zero classes in confusion matrix for clarity
unique_classes = np.unique(np.concatenate([comparison_true_classes, comparison_pred_classes]))
cm_comparison_filtered = confusion_matrix(comparison_true_classes, comparison_pred_classes, labels=unique_classes)
sns.heatmap(cm_comparison_filtered, annot=True, fmt='d', cmap='Greens', ax=ax4, cbar=True)
ax4.set_xlabel('Predicted Score')
ax4.set_ylabel('True Score')
ax4.set_title(f'Comparison Score (Classification)\nAccuracy: {acc_comparison:.3f}, F1: {f1_comp:.3f}, MAE: {mae_comparison:.3f}')
ax4.set_xticklabels([f'{i+1}' for i in unique_classes], rotation=0)
ax4.set_yticklabels([f'{i+1}' for i in unique_classes], rotation=0)

# ===== COMPARISON SCORE (Ordinal Error Distribution) =====
ax5 = fig.add_subplot(gs[2, :])
errors = comparison_pred_original - comparison_true_original
error_bins = np.arange(-9.5, 10.5, 1)
ax5.hist(errors, bins=error_bins, color='orange', alpha=0.7, edgecolor='black')
ax5.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction')
ax5.set_xlabel('Prediction Error (Predicted - True)', fontsize=11)
ax5.set_ylabel('Frequency', fontsize=11)
ax5.set_title('Comparison Score - Prediction Error Distribution', fontsize=12)
ax5.legend()
ax5.grid(True, alpha=0.3, axis='y')

plt.savefig('multitask_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualization saved to 'multitask_results.png'")

In [ ]:
# ========================
# 8. Multi-Task Explainability for 4 ADOS Metrics
# Using Integrated Gradients + Feature Ablation
# ========================
import numpy as np
import json
import torch
import torch.nn as nn
from typing import Dict, List, Tuple
import matplotlib.pyplot as plt

# ========================
# Joint Names Mapping (COCO format)
# ========================
COCO_JOINTS = [
    "nose", "neck", "right_shoulder", "right_elbow", "right_wrist",
    "left_shoulder", "left_elbow", "left_wrist", "right_hip", "right_knee",
    "right_ankle", "left_hip", "left_knee", "left_ankle", "right_eye",
    "left_eye", "right_ear", "left_ear", "pelvis", "thorax",
    "upper_neck", "head_top", "right_big_toe", "left_big_toe"
]

# ========================
# Multi-Task Explainability Module
# ========================
class MultiTaskADOSExplainer:
    def __init__(self, model, target_scalers, demographic_scaler, fps=30, device='cpu'):
        """
        Initialize explainer for 4-task ADOS model with demographic inputs
        """
        self.model = model.to(device)
        self.device = device
        self.model.eval()
        self.target_scalers = target_scalers
        self.demographic_scaler = demographic_scaler
        self.baseline_sequence = None
        self.fps = fps
        
        self.target_names = ['Severity', 'Social Affect', 'RRB', 'Comparison Score']
    
    def compute_integrated_gradients(self, x_seq: torch.Tensor, x_demo: torch.Tensor,
                                     seq_len: torch.Tensor, task_idx: int = 0, 
                                     steps: int = 50) -> Tuple[np.ndarray, np.ndarray, float, float]:
        """
        Compute Integrated Gradients for a specific task
        Returns attributions for both sequence and demographic features
        """
        if x_seq.dim() == 2:
            x_seq = x_seq.unsqueeze(0)
        if x_demo.dim() == 1:
            x_demo = x_demo.unsqueeze(0)
        if seq_len.dim() == 0:
            seq_len = seq_len.unsqueeze(0)
        
        x_seq = x_seq.to(self.device)
        x_demo = x_demo.to(self.device)
        seq_len = seq_len.to(self.device)
        
        # Create baselines
        if self.baseline_sequence is None or self.baseline_sequence.shape != x_seq.shape:
            self.baseline_sequence = torch.zeros_like(x_seq).to(self.device)
        baseline_demo = torch.zeros_like(x_demo).to(self.device)
        
        # Get predictions
        with torch.no_grad():
            baseline_out = self.model(self.baseline_sequence, baseline_demo, seq_len)
            actual_out = self.model(x_seq, x_demo, seq_len)
            
            # Extract predictions based on task type
            if task_idx == 0:  # Severity (classification)
                baseline_pred = torch.argmax(baseline_out['severity_logits'], dim=1).item()
                actual_pred = torch.argmax(actual_out['severity_logits'], dim=1).item()
            elif task_idx == 1:  # Social Affect (regression)
                baseline_pred = baseline_out['social_affect'].squeeze().item()
                actual_pred = actual_out['social_affect'].squeeze().item()
            elif task_idx == 2:  # RRB (regression)
                baseline_pred = baseline_out['rrb'].squeeze().item()
                actual_pred = actual_out['rrb'].squeeze().item()
            elif task_idx == 3:  # Comparison Score (classification)
                baseline_pred = torch.argmax(baseline_out['comparison_logits'], dim=1).item()
                actual_pred = torch.argmax(actual_out['comparison_logits'], dim=1).item()
        
        # Compute integrated gradients
        integrated_grads_seq = torch.zeros_like(x_seq)
        integrated_grads_demo = torch.zeros_like(x_demo)
        
        for step in range(steps):
            alpha = (step + 1) / steps
            interpolated_seq = self.baseline_sequence + alpha * (x_seq - self.baseline_sequence)
            interpolated_demo = baseline_demo + alpha * (x_demo - baseline_demo)
            
            interpolated_seq.requires_grad_(True)
            interpolated_demo.requires_grad_(True)
            
            output = self.model(interpolated_seq, interpolated_demo, seq_len)
            
            # Extract task output based on task type
            if task_idx == 0:  # Severity
                task_output = output['severity_logits'][:, 0]  # Use logit for class 0
            elif task_idx == 1:  # Social Affect
                task_output = output['social_affect'].squeeze()
            elif task_idx == 2:  # RRB
                task_output = output['rrb'].squeeze()
            elif task_idx == 3:  # Comparison Score
                task_output = output['comparison_logits'][:, 0]  # Use logit for class 0
            
            self.model.zero_grad()
            task_output.sum().backward()
            
            integrated_grads_seq += interpolated_seq.grad
            integrated_grads_demo += interpolated_demo.grad
        
        # Average and scale
        integrated_grads_seq = integrated_grads_seq / steps
        integrated_grads_demo = integrated_grads_demo / steps
        
        attributions_seq = (x_seq - self.baseline_sequence) * integrated_grads_seq
        attributions_demo = (x_demo - baseline_demo) * integrated_grads_demo
        
        return (attributions_seq[0].cpu().detach().numpy(), 
                attributions_demo[0].cpu().detach().numpy(),
                actual_pred, baseline_pred)

    def compute_temporal_contributions(self, attributions_seq: np.ndarray, 
                                      seq_len: int, 
                                      window_size_seconds: float = 1.0) -> List[Dict]:
        """
        Analyze temporal contributions by aggregating frame attributions into time windows
        
        Args:
            attributions_seq: Frame-level attributions [num_frames, num_features]
            seq_len: Actual sequence length (excluding padding)
            window_size_seconds: Size of temporal window in seconds
            
        Returns:
            List of dictionaries with temporal segment contributions
        """
        # Only consider non-padded frames
        attributions_seq = attributions_seq[:seq_len, :]
        
        # Calculate window size in frames
        window_size_frames = int(window_size_seconds * self.fps)
        if window_size_frames < 1:
            window_size_frames = 1
        
        # Aggregate attributions per frame (sum absolute values across all features)
        frame_contributions = np.sum(np.abs(attributions_seq), axis=1)
        
        # Create temporal windows
        num_windows = int(np.ceil(seq_len / window_size_frames))
        temporal_segments = []
        
        for i in range(num_windows):
            start_frame = i * window_size_frames
            end_frame = min((i + 1) * window_size_frames, seq_len)
            
            # Time in seconds
            start_time = start_frame / self.fps
            end_time = end_frame / self.fps
            
            # Aggregate contributions for this window
            window_contribution = np.sum(frame_contributions[start_frame:end_frame])
            
            # Get mean attribution sign (positive or negative influence)
            window_raw_attribution = np.sum(attributions_seq[start_frame:end_frame, :])
            
            temporal_segments.append({
                'start_time': float(start_time),
                'end_time': float(end_time),
                'start_frame': int(start_frame),
                'end_frame': int(end_frame),
                'contribution': float(window_contribution),
                'influence_direction': 'positive' if window_raw_attribution > 0 else 'negative',
                'raw_attribution': float(window_raw_attribution)
            })
        
        # Sort by contribution magnitude
        temporal_segments = sorted(temporal_segments, 
                                  key=lambda x: abs(x['contribution']), 
                                  reverse=True)
        
        return temporal_segments

    def compute_confidence(self, x_seq: torch.Tensor, x_demo: torch.Tensor, 
                          seq_len: torch.Tensor, task_idx: int, n_samples: int = 30) -> Dict:
        """
        Compute prediction confidence using Monte Carlo Dropout
        Higher confidence means more certain prediction
        
        Returns:
            Dict with confidence score (0-100%), std_dev, and confidence_level
        """
        self.model.train()  # Enable dropout for uncertainty estimation
        predictions = []
        
        with torch.no_grad():
            for _ in range(n_samples):
                out = self.model(x_seq.to(self.device), x_demo.to(self.device), seq_len.to(self.device))
                
                # Extract prediction based on task type and convert to original scale
                if task_idx == 0:  # Severity (classification)
                    pred_class = torch.argmax(out['severity_logits'], dim=1).item()
                    pred_original = severity_inverse_mapping[pred_class]
                elif task_idx == 1:  # Social Affect (regression)
                    pred_scaled = out['social_affect'].squeeze().item()
                    pred_original = self.target_scalers[1].inverse_transform([[pred_scaled]])[0, 0]
                elif task_idx == 2:  # RRB (regression)
                    pred_scaled = out['rrb'].squeeze().item()
                    pred_original = self.target_scalers[2].inverse_transform([[pred_scaled]])[0, 0]
                elif task_idx == 3:  # Comparison Score (classification)
                    pred_class = torch.argmax(out['comparison_logits'], dim=1).item()
                    pred_original = pred_class + 1  # 0-9 -> 1-10
                
                predictions.append(pred_original)
        
        self.model.eval()  # Back to eval mode
        
        predictions = np.array(predictions)
        mean_pred = np.mean(predictions)
        std_pred = np.std(predictions)
        
        # Compute confidence score (inverse of coefficient of variation)
        # Higher std = lower confidence
        if mean_pred != 0:
            cv = abs(std_pred / mean_pred)  # Coefficient of variation
            confidence = max(0, min(100, 100 * (1 - cv)))  # Convert to 0-100% scale
        else:
            confidence = 50.0  # Default for zero predictions
        
        # Determine confidence level
        if confidence >= 80:
            confidence_level = "High"
        elif confidence >= 60:
            confidence_level = "Medium"
        else:
            confidence_level = "Low"
        
        return {
            'confidence_score': float(confidence),
            'confidence_level': confidence_level,
            'prediction_std': float(std_pred),
            'prediction_mean': float(mean_pred)
        }
            
    
    def explain_all_tasks(self, x_seq: torch.Tensor, x_demo: torch.Tensor, seq_len: torch.Tensor = None) -> Dict:
        """Generate comprehensive explanation for all 4 tasks"""
        if x_seq.dim() == 2:
            x_seq = x_seq.unsqueeze(0)
        if x_demo.dim() == 1:
            x_demo = x_demo.unsqueeze(0)
        if seq_len is None:
            seq_len = torch.tensor([x_seq.shape[1]], dtype=torch.long)
        elif seq_len.dim() == 0:
            seq_len = seq_len.unsqueeze(0)
        
        seq_length = seq_len.item()
        video_length = seq_length / self.fps

        print(f"Computing explanations for 4 ADOS metrics (Video: {video_length:.1f}s)...")

        
        # Get all predictions
        with torch.no_grad():
            x_seq_device = x_seq.to(self.device)
            x_demo_device = x_demo.to(self.device)
            seq_len_device = seq_len.to(self.device)
            all_outputs = self.model(x_seq_device, x_demo_device, seq_len_device)
        
        # Convert to original scale
        # Model now returns dict: {'severity_logits', 'social_affect', 'rrb', 'comparison_logits'}
        predictions = {}
        
        # Severity (classification) - get predicted class
        severity_class = torch.argmax(all_outputs['severity_logits'], dim=1).item()
        predictions['severity'] = float(severity_inverse_mapping[severity_class])
        
        # Social Affect (regression)
        social_affect_scaled = all_outputs['social_affect'].squeeze().item()
        predictions['social_affect'] = float(
            self.target_scalers[1].inverse_transform([[social_affect_scaled]])[0, 0]
        )
        
        # RRB (regression)
        rrb_scaled = all_outputs['rrb'].squeeze().item()
        predictions['rrb'] = float(
            self.target_scalers[2].inverse_transform([[rrb_scaled]])[0, 0]
        )
        
        # Comparison Score (classification) - get predicted class
        comparison_class = torch.argmax(all_outputs['comparison_logits'], dim=1).item()
        predictions['comparison_score'] = int(comparison_class + 1)  # 0-9 -> 1-10
        
        # Get demographic values (original scale)
        demo_original = self.demographic_scaler.inverse_transform(x_demo.cpu().numpy())[0]
        predictions['age_input'] = float(demo_original[0])
        predictions['gender_input'] = int(demo_original[1])
        
        # Compute integrated gradients for all tasks
        print("Computing explanations for 4 ADOS metrics...")
        
        explanations_per_task = {}
        for task_idx in range(4):
            task_name = self.target_names[task_idx]
            print(f"  Analyzing {task_name}...")
            
            attr_seq, attr_demo, pred, baseline = self.compute_integrated_gradients(
                x_seq, x_demo, seq_len, task_idx
            )
            # Compute confidence for this task
            print(f"    Computing confidence...")
            confidence_info = self.compute_confidence(x_seq, x_demo, seq_len, task_idx)
            attr_seq, attr_demo, pred, baseline = self.compute_integrated_gradients(
                x_seq, x_demo, seq_len, task_idx
            )
            
            
            # Get top contributing joints
            joint_contributions = self._compute_joint_contributions(attr_seq, seq_length)
            top_joints_positive = sorted(
                [(j, c) for j, c in joint_contributions.items() if c > 0],
                key=lambda x: abs(x[1]), reverse=True
            )
            top_joints_negative = sorted(
                [(j, c) for j, c in joint_contributions.items() if c < 0],
                key=lambda x: abs(x[1]), reverse=True
            )
            
            # Compute temporal contributions
            temporal_segments = self.compute_temporal_contributions(
                attr_seq, seq_length, window_size_seconds=1.0
            )
            
            # Separate positive and negative temporal influences
            positive_segments = [s for s in temporal_segments if s['influence_direction'] == 'positive'][:5]
            negative_segments = [s for s in temporal_segments if s['influence_direction'] == 'negative'][:5]
            # Demographic contributions
            demo_contrib = {
                'age_contribution': float(attr_demo[0]),
                'gender_contribution': float(attr_demo[1])
            }
            
            explanations_per_task[task_name] = {
                'prediction': float(pred),
                'baseline': float(baseline),
                'confidence': confidence_info,
                'joints': {
                    'positive_contributors': [
                        {'joint': j, 'contribution': float(c)} 
                        for j, c in top_joints_positive
                    ],
                    'negative_contributors': [
                        {'joint': j, 'contribution': float(c)} 
                        for j, c in top_joints_negative
                    ]
                },
                'temporal_segments': {
                    'positive_segments': positive_segments,
                    'negative_segments': negative_segments,
                    'all_segments': temporal_segments
                },
                'demographic_contributions': demo_contrib,
                'total_sequence_attribution': float(np.sum(np.abs(attr_seq))),
                'total_demographic_attribution': float(np.sum(np.abs(attr_demo)))
            }
        
        explanation = {
            'predictions': predictions,
            'video_metadata': {
                'duration_seconds': float(video_length),
                'num_frames': int(seq_length),
                'fps': self.fps
            },
            'task_explanations': explanations_per_task,
            'summary': self._generate_summary(predictions, explanations_per_task)
        }
        
        return explanation
    
    def _compute_joint_contributions(self, attributions: np.ndarray, seq_len: int) -> Dict[str, float]:
        """Aggregate contributions per joint"""
        attributions = attributions[:seq_len, :]
        joint_attrs = attributions[:, :48].reshape(seq_len, 24, 2)
        joint_contributions = np.sum(joint_attrs, axis=(0, 2))
        
        return {COCO_JOINTS[i]: float(joint_contributions[i]) for i in range(len(COCO_JOINTS))}
    
    def _generate_summary(self, predictions: Dict, task_explanations: Dict) -> str:
        """Generate human-readable summary"""
        summary = "Multi-Task ADOS Assessment (Using Age & Gender as Inputs):\\n\\n"
        
        summary += f"PREDICTIONS:\\n"
        summary += f"  Severity: {predictions['severity']:.1f}\\n"
        summary += f"  Social Affect: {predictions['social_affect']:.1f}\\n"
        summary += f"  RRB: {predictions['rrb']:.1f}\\n"
        summary += f"  Comparison Score: {predictions['comparison_score']:.1f}\\n\\n"
        
        summary += f"INPUT DEMOGRAPHICS:\\n"
        summary += f"  Age: {predictions['age_input']:.1f} years\\n"
        summary += f"  Gender: {predictions['gender_input']}\\n\\n"
        
        # Key findings
        for task_name in ['Severity', 'Social Affect', 'RRB', 'Comparison Score']:
            if task_name in task_explanations:
                exp = task_explanations[task_name]
                top_joint = exp['joints']['positive_contributors'][0] if exp['joints']['positive_contributors'] else {'joint': 'N/A', 'contribution': 0}
                top_time = exp['temporal_segments']['positive_segments'][0] if exp['temporal_segments']['positive_segments'] else None
                demo_total = sum(abs(v) for v in exp['demographic_contributions'].values())
                summary += f"{task_name}: Top joint={top_joint['joint']} ({top_joint['contribution']:+.2f}), "
                summary += f"Demographic impact={demo_total:.2f}\\n"
        
        return summary
    
    def visualize_explanation(self, explanation: Dict, save_path: str = None):
        """Visualize multi-task predictions and explanations"""
        fig = plt.figure(figsize=(18, 14))
        gs = fig.add_gridspec(4, 3, hspace=0.4, wspace=0.3)
        
        preds = explanation['predictions']
        video_meta = explanation['video_metadata']
        
        preds = explanation['predictions']
        
        fig.suptitle(
            f"Multi-Task ADOS Prediction\\n"
            f"Age: {preds['age_input']:.0f} | Gender: {preds['gender_input']} | "
            f"Severity: {preds['severity']:.1f} | Social Affect: {preds['social_affect']:.1f}"
            f"Video: {video_meta['duration_seconds']:.1f}s ({video_meta['num_frames']} frames)",    
            fontsize=14, fontweight='bold'
        )
        
        # Plot 1: All predictions
        ax1 = fig.add_subplot(gs[0, :])
        pred_values = [preds['severity'], preds['social_affect'], preds['rrb'], preds['comparison_score']]
        pred_labels = ['Severity', 'Social\\nAffect', 'RRB', 'Comparison\\nScore']
        
        bars = ax1.bar(pred_labels, pred_values, color='steelblue', alpha=0.7)
        ax1.set_ylabel('Score')
        ax1.set_title('Predicted ADOS Metrics (Demographics as Inputs)')
        ax1.grid(axis='y', alpha=0.3)
        
        # Plots 2-9: Joint and temporal contributions for each task
        task_names = ['Severity', 'Social Affect', 'RRB']
        positions = [(1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2), (3, 0), (3, 1), (3, 2)]
        
        for task_idx, task_name in enumerate(task_names):
            # Joint contributions
            ax_joint = fig.add_subplot(gs[task_idx + 1, 0])
            task_exp = explanation['task_explanations'][task_name]
            
            all_joints = (task_exp['joints']['positive_contributors'][:3] + 
                         task_exp['joints']['negative_contributors'][:2])
            joints = [j['joint'] for j in all_joints]
            contribs = [j['contribution'] for j in all_joints]
            colors = ['#d62728' if c > 0 else '#2ca02c' for c in contribs]
            
            ax_joint.barh(joints, contribs, color=colors, alpha=0.7)
            ax_joint.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
            ax_joint.set_xlabel('Contribution')
            ax_joint.set_title(f'{task_name}: Top Joints\nPred={task_exp["prediction"]:.2f}', fontsize=10)
            ax_joint.invert_yaxis()
            
            # Temporal contributions
            ax_temporal = fig.add_subplot(gs[task_idx + 1, 1:])
            
            pos_segs = task_exp['temporal_segments']['positive_segments'][:5]
            neg_segs = task_exp['temporal_segments']['negative_segments'][:3]
            
            y_pos = 0
            for seg in pos_segs:
                width = seg['end_time'] - seg['start_time']
                ax_temporal.barh(y_pos, width, left=seg['start_time'], 
                               color='#d62728', alpha=0.6, height=0.8)
                ax_temporal.text(seg['start_time'] + width/2, y_pos, 
                               f"{seg['contribution']:.1f}", 
                               ha='center', va='center', fontsize=8)
                y_pos += 1
            
            y_neg = -1
            for seg in neg_segs:
                width = seg['end_time'] - seg['start_time']
                ax_temporal.barh(y_neg, width, left=seg['start_time'], 
                               color='#2ca02c', alpha=0.6, height=0.8)
                ax_temporal.text(seg['start_time'] + width/2, y_neg, 
                               f"{seg['contribution']:.1f}", 
                               ha='center', va='center', fontsize=8)
                y_neg -= 1
            
            ax_temporal.axhline(y=-0.5, color='black', linestyle='--', linewidth=0.5)
            ax_temporal.set_xlabel('Time (seconds)')
            ax_temporal.set_title(f'{task_name}: Critical Time Segments\n(Red=Problematic, Green=Strength)', fontsize=10)
            ax_temporal.set_xlim(0, video_meta['duration_seconds'])
            ax_temporal.set_ylim(y_neg - 1, y_pos)
            ax_temporal.grid(axis='x', alpha=0.3)
            
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"✅ Visualization saved to {save_path}")
        
        plt.show()

print("✅ Multi-Task Explainability Module Ready (4 ADOS Metrics + Demographics)")

In [ ]:
# ========================
# 9. Run Multi-Task Explainability (4 ADOS Metrics)
# ========================
def explain_multitask_samples(model, val_loader, target_scalers, fps, demographic_scaler, 
                               num_samples=3, device='cpu'):
    """Generate explanations for multi-task model with demographic inputs"""
    explainer = MultiTaskADOSExplainer(
    model=model,
    target_scalers=target_scalers,
    demographic_scaler=demographic_scaler,
    fps=fps,  # Now correctly passed as fps (not confused with demographic_scaler)
    device=device
)

    explanations = []
    
    print(f"\n{'='*70}")
    print(f"Generating Explanations for {num_samples} samples...")
    print(f"{'='*70}\n")
    
    sample_count = 0
    for batch in val_loader:
        x_seq, x_demo, y_sev_class, y_sa_reg, y_rrb_reg, y_comp_class, y_orig, lens = batch
        
        for i in range(x_seq.shape[0]):
            if sample_count >= num_samples:
                break
            
            seq = x_seq[i]
            demo = x_demo[i]
            y_true_orig = y_orig[i].numpy()
            seq_len = lens[i]
            # Get demographics in original scale
            demo_orig = demographic_scaler.inverse_transform(demo.numpy().reshape(1, -1))[0]
            
            print(f"\n{'='*70}")
            print(f"SAMPLE {sample_count + 1}/{num_samples}")
            print(f"Age: {demo_orig[0]:.1f}, Gender: {int(demo_orig[1])}")
            print(f"{'='*70}")
            
            # Generate explanation
            explanation = explainer.explain_all_tasks(
                x_seq=seq,
                seq_len=seq_len,
                x_demo=demo  # or None if your explainer doesn't need fps
            )

            
            # Add ground truth
            explanation['ground_truth'] = {
                'severity': float(y_true_orig[0]),
                'social_affect': float(y_true_orig[1]),
                'rrb': float(y_true_orig[2]),
                'comparison_score': float(y_true_orig[3])
            }
            
            # Print summary
            print(explanation['summary'])
            
            print("\nGround Truth vs Predictions:")
            for i, task in enumerate(['severity', 'social_affect', 'rrb', 'comparison_score']):
                true_val = explanation['ground_truth'][task]
                pred_val = explanation['predictions'][task]
                error = abs(true_val - pred_val)
                print(f"  {task}: True={true_val:.2f}, Pred={pred_val:.2f}, Error={error:.2f}")

            # Print temporal insights
            print("\n" + "="*80)
            print("TEMPORAL INSIGHTS (Critical Time Segments):")
            print("="*80)
            for task_name in ['Severity', 'Social Affect', 'RRB']:
                task_exp = explanation['task_explanations'][task_name]
                print(f"\n{task_name}:")
                
                print("  Problematic segments (increase score):")
                for seg in task_exp['temporal_segments']['positive_segments'][:3]:
                    print(f"    {seg['start_time']:.1f}-{seg['end_time']:.1f}s: contribution={seg['contribution']:.2f}")
                
                print("  Strength segments (decrease score):")
                for seg in task_exp['temporal_segments']['negative_segments'][:2]:
                    print(f"    {seg['start_time']:.1f}-{seg['end_time']:.1f}s: contribution={seg['contribution']:.2f}")
            
            explanations.append(explanation)
            
            # Visualize
            explainer.visualize_explanation(
                explanation, 
                save_path=f'multitask_explanation_{sample_count+1}.png'
            )
            
            sample_count += 1
        
        if sample_count >= num_samples:
            break
    
    # Save for therapy engine
    if explanations:
        with open('multitask_therapy_input.json', 'w') as f:
            json.dump(explanations[0], f, indent=2, default=str)
        print(f"\n✅ Saved therapy engine input to multitask_therapy_input.json")
    
    return explanations

# Run the explainer
explanations = explain_multitask_samples(
    model=model,
    val_loader=val_loader,
    target_scalers=target_scalers,
    fps=30,
    demographic_scaler=demographic_scaler,
    num_samples=3,
    device='cpu'
)

print("\n" + "="*70)
print("MULTI-TASK EXPLAINABILITY COMPLETE!")
print("="*70)
print(f"\n✅ Generated {len(explanations)} complete explanations")
print("✅ Each explanation shows:")
print("   - Predictions for 4 ADOS metrics")
print("   - How age and gender influence each prediction")
print("   - Which body joints contribute most to each score")
print("   - Temporal patterns and behavioral indicators")

In [ ]:
# ========================
# 10. Helper Functions for Inference (UPDATED FOR MIXED TASKS)
# ========================

def predict_single_sample(model, sequence_folder_path, age, gender_value, 
                          feature_scaler, demographic_scaler, target_scalers, 
                          gender_encoder, severity_inverse_mapping, max_len=100):
    """
    Make prediction on a single video sequence with demographic information (UPDATED)
    
    Args:
        model: Trained multi-task model
        sequence_folder_path: Path to folder containing .npz frame files
        age: Patient's age (years)
        gender_value: Patient's gender (original value, e.g., 'M', 'F')
        feature_scaler: Fitted StandardScaler for skeletal features
        demographic_scaler: Fitted StandardScaler for demographics
        target_scalers: List of scalers [None, social_affect_scaler, rrb_scaler, None]
        gender_encoder: LabelEncoder for gender
        severity_inverse_mapping: Dict to map class indices back to severity values
        max_len: Maximum sequence length
    
    Returns:
        Dictionary with all predictions in original scale
    """
    # Load and process sequence
    sequence = load_npz_sequence(sequence_folder_path)
    
    if sequence.shape[0] == 0:
        return {"error": "Empty sequence"}
    
    # Preprocess skeletal features
    original_length = min(sequence.shape[0], max_len)
    sequence = normalize_skeleton(sequence)
    sequence = pad_sequence(sequence, max_len)
    features = compute_enhanced_features(sequence)
    
    # Normalize skeletal features
    features_flat = features.reshape(-1, features.shape[-1])
    features_normalized = feature_scaler.transform(features_flat)
    features_normalized = features_normalized.reshape(features.shape)
    
    # Encode and normalize demographic features
    gender_encoded = gender_encoder.transform([gender_value])[0]
    demo_features = np.array([[age, gender_encoded]])
    demo_normalized = demographic_scaler.transform(demo_features)
    
    # Convert to tensors
    x_seq = torch.tensor(features_normalized, dtype=torch.float32).unsqueeze(0)
    x_demo = torch.tensor(demo_normalized, dtype=torch.float32).squeeze(0)
    seq_len = torch.tensor([original_length], dtype=torch.long)
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(x_seq, x_demo, seq_len)
    
    # Extract predictions
    # Severity: classification - get predicted class
    severity_class = torch.argmax(outputs['severity_logits'], dim=1).item()
    severity_value = severity_inverse_mapping[severity_class]
    
    # Social Affect: regression - inverse transform
    social_affect_scaled = outputs['social_affect'].squeeze().item()
    social_affect_value = target_scalers[1].inverse_transform([[social_affect_scaled]])[0, 0]
    
    # RRB: regression - inverse transform
    rrb_scaled = outputs['rrb'].squeeze().item()
    rrb_value = target_scalers[2].inverse_transform([[rrb_scaled]])[0, 0]
    
    # Comparison Score: classification - get predicted class (0-9) and add 1 to get (1-10)
    comparison_class = torch.argmax(outputs['comparison_logits'], dim=1).item()
    comparison_value = comparison_class + 1
    
    predictions = {
        'severity': float(severity_value),
        'social_affect': float(social_affect_value),
        'rrb': float(rrb_value),
        'comparison_score': int(comparison_value),
        'input_age': age,
        'input_gender': gender_value
    }
    
    return predictions

def batch_predict_and_save(model, root_folder, metadata_file, feature_scaler, 
                           demographic_scaler, target_scalers, gender_encoder, 
                           severity_inverse_mapping, output_file='predictions.csv'):
    """
    Make predictions on all sequences and save to CSV (UPDATED)
    """
    results = []
    
    df = pd.read_excel(metadata_file)
    
    # Convert age to years
    def convert_age_to_years(age_str):
        try:
            if isinstance(age_str, (int, float)):
                return float(age_str)
            age_str = str(age_str).strip()
            parts = age_str.split(',')
            years = 0
            months = 0
            for part in parts:
                part = part.strip().upper()
                if 'Y' in part:
                    years = int(part.replace('Y', '').strip())
                elif 'M' in part:
                    months = int(part.replace('M', '').strip())
            return years + (months / 12.0)
        except Exception as e:
            return None
    
    id_to_info = {}
    for idx, row in df.iterrows():
        age_value = convert_age_to_years(row['Chronological Age'])
        if age_value is not None:
            id_to_info[row['ID#']] = {
                'age': age_value,
                'gender': row['Gender'],
                'severity': row['Severity of Autism  '],
                'social_affect': row['Social Affect Total '],
                'rrb': row['Restricted and Repetitive Behavior (RRB) Total'],
                'comparison': row['ADOS Comparison Score (1-10) <5  not very autistic. ASD people usually fall 5-10. 8-10=Severe, 5-7=moderate, 1-4=mild']
            }
    
    for action in os.listdir(root_folder):
        print(f"Loading Action: {action}")
        action_path = os.path.join(root_folder, action)
        if not os.path.isdir(action_path):
            continue
        
        for trial in os.listdir(action_path):
            trial_path = os.path.join(action_path, trial)
            if not os.path.isdir(trial_path):
                continue
            
            try:
                patient_id = int(trial.split('_')[1])
            except (IndexError, ValueError):
                continue
            
            if patient_id not in id_to_info:
                continue
            
            # Get demographic info
            patient_info = id_to_info[patient_id]
            age = patient_info['age']
            gender = patient_info['gender']
            
            # Make prediction
            preds = predict_single_sample(
                model, trial_path, age, gender,
                feature_scaler, demographic_scaler, 
                target_scalers, gender_encoder,
                severity_inverse_mapping
            )
            
            if 'error' in preds:
                continue
            
            # Build result
            result = {
                'patient_id': patient_id,
                'action': action,
                'trial': trial,
                'age_input': age,
                'gender_input': gender,
                **{k: v for k, v in preds.items() if k not in ['input_age', 'input_gender']}
            }
            
            # Add ground truth
            result['true_severity'] = patient_info['severity']
            result['true_social_affect'] = patient_info['social_affect']
            result['true_rrb'] = patient_info['rrb']
            result['true_comparison'] = patient_info['comparison']
            
            results.append(result)
    
    # Save to CSV
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_file, index=False)
    
    print(f"✅ Saved {len(results)} predictions to {output_file}")
    
    return results_df

print("✅ Helper functions for inference ready (UPDATED FOR MIXED TASKS)")
print("\nUsage:")
print("  predict_single_sample(model, path, age, gender, ..., severity_inverse_mapping) - Predict on one video")
print("  batch_predict_and_save(model, ..., severity_inverse_mapping) - Predict on entire dataset")
print("\nExample:")
print("  preds = predict_single_sample(model, 'path/to/video', age=8, gender_value='M', ...)")
print("  # Returns: {'severity': 2 or 3, 'social_affect': 8.5, 'rrb': 3.8, 'comparison_score': 7 (1-10)}")

In [ ]:
with open('multitask_therapy_input.json', 'r') as f:
        print(f.read())

# 3D Model

In [ ]:
# ========================
# 3D Model - Step 1: Enhanced preprocessing for 3D coordinates
# ========================
import numpy as np
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

def load_npz_sequence_3d(folder_path):
    """Load 3D skeleton sequence (24 joints with x, y, z coordinates)"""
    frame_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.npz')])
    sequence = []
    for f in frame_files:
        data = np.load(os.path.join(folder_path, f))['coordinates']
        if data.shape[0] > 0:
            person_data = data[0]  # Take first person
            sequence.append(person_data)
    if not sequence:
        return np.empty((0, 24, 3))  # 3D coordinates
    return np.array(sequence)

def normalize_skeleton_3d(sequence):
    """Improved normalization for 3D skeleton with stability"""
    if sequence.shape[0] == 0:
        return sequence
    
    # Center around hip midpoint (3D)
    center = (sequence[:, 8:9, :] + sequence[:, 11:12, :]) / 2
    sequence = sequence - center
    
    # Normalize by torso length for scale invariance (3D distance)
    torso_length = np.linalg.norm(
        sequence[:, 1, :] - (sequence[:, 8, :] + sequence[:, 11, :]) / 2,
        axis=1, keepdims=True
    )
    torso_length = np.maximum(torso_length, 1e-6)  # Avoid division by zero
    sequence = sequence / (torso_length[:, :, np.newaxis] + 1e-6)
    
    return sequence

def pad_sequence_3d(seq, max_len=500):
    """Pad 3D sequence to fixed length"""
    num_frames, num_joints, _ = seq.shape
    if num_frames >= max_len:
        return seq[:max_len]
    else:
        pad_len = max_len - num_frames
        padding = np.zeros((pad_len, num_joints, 3))  # 3D padding
        return np.concatenate([seq, padding], axis=0)

def compute_enhanced_features_3d(sequence):
    """
    Enhanced feature extraction for 3D skeleton data
    Same behavioral signals as 2D but with 3D coordinates
    """
    num_frames = sequence.shape[0]
    
    # 1. Flattened joint positions (72 features: 24 joints * 3 coords)
    flat_joints = sequence.reshape(num_frames, -1)
    
    # 2. Velocity (72 features)
    velocity = np.zeros_like(flat_joints)
    velocity[1:] = flat_joints[1:] - flat_joints[:-1]
    
    # 3. Acceleration (72 features)
    acceleration = np.zeros_like(flat_joints)
    acceleration[1:] = velocity[1:] - velocity[:-1]
    
    # 4. Key joint distances (behavioral indicators) - 3D distances
    # Hand-hand distance
    left_wrist = sequence[:, 7, :]
    right_wrist = sequence[:, 4, :]
    dist_hands = np.linalg.norm(left_wrist - right_wrist, axis=1, keepdims=True)
    
    # Elbow-elbow distance
    left_elbow = sequence[:, 6, :]
    right_elbow = sequence[:, 3, :]
    dist_elbows = np.linalg.norm(left_elbow - right_elbow, axis=1, keepdims=True)
    
    # Hand-to-body distances (self-stimulatory behaviors)
    neck = sequence[:, 1, :]
    dist_left_hand_body = np.linalg.norm(left_wrist - neck, axis=1, keepdims=True)
    dist_right_hand_body = np.linalg.norm(right_wrist - neck, axis=1, keepdims=True)
    
    # 5. Movement magnitude (overall activity level)
    movement_mag = np.linalg.norm(velocity.reshape(num_frames, -1), axis=1, keepdims=True)
    
    # 6. Symmetry features (left-right coordination)
    left_shoulder = sequence[:, 5, :]
    right_shoulder = sequence[:, 2, :]
    shoulder_symmetry = np.linalg.norm(
        (left_wrist - left_shoulder) - (right_wrist - right_shoulder),
        axis=1, keepdims=True
    )
    
    # Combine all features (72 + 72 + 72 + 1 + 1 + 1 + 1 + 1 + 1 = 222 features)
    features = np.concatenate([
        flat_joints,           # 72
        velocity,              # 72
        acceleration,          # 72
        dist_hands,            # 1
        dist_elbows,           # 1
        dist_left_hand_body,   # 1
        dist_right_hand_body,  # 1
        movement_mag,          # 1
        shoulder_symmetry      # 1
    ], axis=1)
    
    return features

print("✅ 3D preprocessing functions ready")
print("Key differences from 2D:")
print("  - Input: 24 joints × 3 coordinates (x, y, z)")
print("  - Features: 222 per frame (vs 150 in 2D)")
print("  - Distance calculations use 3D Euclidean norm")

In [ ]:
# ========================
# 3D Model - Step 2: Load 3D dataset with demographics
# ========================
root_folder_3d = "/kaggle/input/3d-coords/ROMP_3D_Coordinates"
metadata_file = "/kaggle/input/ados-rating/ADOS_rating.xlsx"
max_len = 100

def convert_age_to_years(age_str):
    """
    Convert age from format like "5Y,10M" to decimal years
    Example: "5Y,10M" -> 5 + 10/12 = 5.833
             "8Y,0M" -> 8.0
             "3Y,6M" -> 3.5
    """
    try:
        # If already a number, return it
        if isinstance(age_str, (int, float)):
            return float(age_str)
        
        # Convert to string and clean
        age_str = str(age_str).strip()
        
        # Split by comma
        parts = age_str.split(',')
        
        years = 0
        months = 0
        
        for part in parts:
            part = part.strip().upper()
            if 'Y' in part:
                years = int(part.replace('Y', '').strip())
            elif 'M' in part:
                months = int(part.replace('M', '').strip())
        
        # Convert to decimal years
        decimal_age = years + (months / 12.0)
        return decimal_age
    
    except Exception as e:
        print(f"Warning: Could not convert age '{age_str}': {e}")
        return None

# Reuse the age conversion function from 2D model
# Load metadata (same as 2D)
df = pd.read_excel(metadata_file)

print("Available columns in ADOS rating file:")
print(df.columns.tolist())
print(f"\nDataset has {len(df)} patients")

# Same target columns as 2D model
target_columns = {
    'severity': 'Severity of Autism  ',
    'age': 'Chronological Age',
    'gender': 'Gender',
    'social_affect': 'Social Affect Total ',
    'rrb': 'Restricted and Repetitive Behavior (RRB) Total',
    'comparison_score': 'ADOS Comparison Score (1-10) <5  not very autistic. ASD people usually fall 5-10. 8-10=Severe, 5-7=moderate, 1-4=mild',
    'overall_total': 'Overall Total '
}

# Create mappings (same as 2D)
id_to_data_3d = {}
for idx, row in df.iterrows():
    patient_id = row['ID#']
    age_value = convert_age_to_years(row[target_columns['age']])
    if age_value is None:
        continue
    id_to_data_3d[patient_id] = {
        'severity': row[target_columns['severity']],
        'age': age_value,
        'gender': row[target_columns['gender']],
        'social_affect': row[target_columns['social_affect']],
        'rrb': row[target_columns['rrb']],
        'comparison_score': row[target_columns['comparison_score']],
        'overall_total': row[target_columns['overall_total']]
    }

# Encode gender (same encoder as 2D for consistency)
gender_encoder_3d = LabelEncoder()
all_genders_3d = [data['gender'] for data in id_to_data_3d.values() if pd.notna(data['gender'])]
gender_encoder_3d.fit(all_genders_3d)
print(f"\nGender encoding: {dict(zip(gender_encoder_3d.classes_, gender_encoder_3d.transform(gender_encoder_3d.classes_)))}")

# Load 3D sequences and labels
sequence_features_3d = []
demographic_features_3d = []
labels_3d = []
sequence_lengths_3d = []
valid_patient_ids_3d = []

print(f"\nLoading 3D sequences from: {root_folder_3d}")

for action in os.listdir(root_folder_3d):
    action_path = os.path.join(root_folder_3d, action)
    print(f"Loading Action: {action}")
    if not os.path.isdir(action_path):
        continue
    for trial in os.listdir(action_path):
        trial_path = os.path.join(action_path, trial)
        if not os.path.isdir(trial_path):
            continue
        try:
            patient_id = int(trial.split('_')[1])
        except (IndexError, ValueError):
            continue
        if patient_id not in id_to_data_3d:
            continue
        
        # Load 3D sequence
        sequence = load_npz_sequence_3d(trial_path)
        if sequence.shape[0] == 0:
            continue
        
        patient_data = id_to_data_3d[patient_id]
        
        # Check for missing values
        if any(pd.isna(patient_data[k]) for k in patient_data.keys()):
            continue
        
        # Encode demographics
        gender_encoded = gender_encoder_3d.transform([patient_data['gender']])[0]
        age_value = patient_data['age']
        
        # Process 3D sequence
        original_length = min(sequence.shape[0], max_len)
        sequence = normalize_skeleton_3d(sequence)
        sequence = pad_sequence_3d(sequence, max_len)
        features = compute_enhanced_features_3d(sequence)
        
        # Store features
        sequence_features_3d.append(features)
        demographic_features_3d.append([age_value, gender_encoded])
        
        # Create label vector (same 4 targets as 2D)
        label_vector = [
            patient_data['severity'],
            patient_data['social_affect'],
            patient_data['rrb'],
            patient_data['comparison_score']
        ]
        labels_3d.append(label_vector)
        sequence_lengths_3d.append(original_length)
        valid_patient_ids_3d.append(patient_id)

sequence_features_3d = np.array(sequence_features_3d)
demographic_features_3d = np.array(demographic_features_3d)
labels_3d = np.array(labels_3d)
sequence_lengths_3d = np.array(sequence_lengths_3d)

print(f"\n3D Sequence features shape: {sequence_features_3d.shape}")
print(f"3D Demographic features shape: {demographic_features_3d.shape}")
print(f"3D Labels shape: {labels_3d.shape}")
print(f"Number of 3D samples: {len(sequence_features_3d)}")

# Print statistics
print("\n" + "="*70)
print("3D MODEL - INPUT FEATURES (Demographics)")
print("="*70)
print(f"Age            - Mean: {demographic_features_3d[:, 0].mean():6.2f}, Std: {demographic_features_3d[:, 0].std():6.2f}")
print(f"Gender (0/1)   - Mean: {demographic_features_3d[:, 1].mean():6.2f}, Distribution: {np.bincount(demographic_features_3d[:, 1].astype(int))}")



In [ ]:
# Clean labels for 3D data (same process as 2D)
labels_3d = np.array(labels_3d, dtype=object).astype(str)

clean_labels_3d = []
for row in labels_3d:
    clean_row = []
    for value in row:
        try:
            clean_value = float(value)
        except:
            clean_value = None
        clean_row.append(clean_value)
    clean_labels_3d.append(clean_row)

clean_labels_3d = np.array(clean_labels_3d, dtype=object)

# Remove rows with None
valid_mask_3d = ~np.any(pd.isna(clean_labels_3d).astype(bool), axis=1)
clean_labels_3d = clean_labels_3d[valid_mask_3d]
sequence_features_3d = sequence_features_3d[valid_mask_3d]
demographic_features_3d = demographic_features_3d[valid_mask_3d]
sequence_lengths_3d = sequence_lengths_3d[valid_mask_3d]

# Convert to float
labels_3d = clean_labels_3d.astype(float)
labels_3d = np.array(labels_3d)

print(f"✅ Cleaned 3D labels: {labels_3d.shape}")
print(f"✅ Final 3D samples: {len(sequence_features_3d)}")

In [ ]:
# ========================
# 3D Model - Step 3: Feature normalization and target scaling (UPDATED FOR CLASSIFICATION)
# ========================
# Normalize 3D skeletal features
original_shape_3d = sequence_features_3d.shape
sequence_reshaped_3d = sequence_features_3d.reshape(-1, sequence_features_3d.shape[-1])

feature_scaler_3d = StandardScaler()
sequence_normalized_3d = feature_scaler_3d.fit_transform(sequence_reshaped_3d)
sequence_features_3d = sequence_normalized_3d.reshape(original_shape_3d)

# Normalize demographic features
demographic_scaler_3d = StandardScaler()
demographic_features_normalized_3d = demographic_scaler_3d.fit_transform(demographic_features_3d)

print("3D Feature scaling:")
print(f"  Skeletal features (222 per frame): StandardScaler applied")
print(f"  Age: StandardScaler applied (as input feature)")
print(f"  Gender: StandardScaler applied (as input feature)")

# Prepare targets (MIXED: 2 classification + 2 regression)
# Target 0: Severity - CLASSIFICATION (2 classes)
# Target 1: Social Affect - REGRESSION
# Target 2: RRB - REGRESSION  
# Target 3: Comparison Score - CLASSIFICATION (10 classes)

# For severity: Convert values to class indices
severity_values_3d = labels_3d[:, 0]
severity_unique_3d = np.unique(severity_values_3d)
print(f"\n3D Severity unique values: {severity_unique_3d}")
severity_mapping_3d = {val: idx for idx, val in enumerate(sorted(severity_unique_3d))}
print(f"3D Severity mapping: {severity_mapping_3d}")
severity_classes_3d = np.array([severity_mapping_3d[val] for val in severity_values_3d])

# For comparison score: Convert to class indices (1-10 -> 0-9)
comparison_values_3d = labels_3d[:, 3]
comparison_unique_3d = np.unique(comparison_values_3d)
print(f"\n3D Comparison score unique values: {comparison_unique_3d}")
comparison_classes_3d = (comparison_values_3d - 1).astype(int)  # 1-10 -> 0-9

# For social affect and RRB: Keep as regression with scaling
social_affect_scaler_3d = StandardScaler()
rrb_scaler_3d = StandardScaler()

social_affect_scaled_3d = social_affect_scaler_3d.fit_transform(labels_3d[:, 1].reshape(-1, 1)).flatten()
rrb_scaled_3d = rrb_scaler_3d.fit_transform(labels_3d[:, 2].reshape(-1, 1)).flatten()

# Store scalers for later use
target_scalers_3d = [None, social_affect_scaler_3d, rrb_scaler_3d, None]  # None for classification tasks

print("\n3D Target preparation:")
print(f"  Severity: Classification ({len(severity_unique_3d)} classes) - {severity_unique_3d}")
print(f"  Social Affect: Regression (Scaled)")
print(f"  RRB: Regression (Scaled)")
print(f"  Comparison Score: Classification (10 classes: 1-10)")

# Convert to torch tensors
X_sequence_3d = torch.tensor(sequence_features_3d, dtype=torch.float32)
X_demographic_3d = torch.tensor(demographic_features_normalized_3d, dtype=torch.float32)

# Create separate tensors for different target types
y_severity_class_3d = torch.tensor(severity_classes_3d, dtype=torch.long)
y_social_affect_reg_3d = torch.tensor(social_affect_scaled_3d, dtype=torch.float32)
y_rrb_reg_3d = torch.tensor(rrb_scaled_3d, dtype=torch.float32)
y_comparison_class_3d = torch.tensor(comparison_classes_3d, dtype=torch.long)

seq_lens_3d = torch.tensor(sequence_lengths_3d, dtype=torch.long)

# Store original labels for evaluation
y_original_3d = torch.tensor(labels_3d, dtype=torch.float32)

# Create dataset with all components
dataset_3d = TensorDataset(
    X_sequence_3d, X_demographic_3d, 
    y_severity_class_3d, y_social_affect_reg_3d, y_rrb_reg_3d, y_comparison_class_3d,
    y_original_3d, seq_lens_3d
)

train_size_3d = int(0.8 * len(dataset_3d))
val_size_3d = len(dataset_3d) - train_size_3d

train_dataset_3d, val_dataset_3d = torch.utils.data.random_split(
    dataset_3d, [train_size_3d, val_size_3d],
    generator=torch.Generator().manual_seed(42)
)

train_loader_3d = DataLoader(train_dataset_3d, batch_size=32, shuffle=True)
val_loader_3d = DataLoader(val_dataset_3d, batch_size=32, shuffle=False)

print(f"\n3D Dataset split:")
print(f"  Training samples: {train_size_3d}")
print(f"  Validation samples: {val_size_3d}")
print(f"  Batch size: 32")

# Store severity mapping for inverse transform
severity_inverse_mapping_3d = {idx: val for val, idx in severity_mapping_3d.items()}

# ========================
# 3D MODEL - HANDLE CLASS IMBALANCE - Calculate class weights
# ========================
from sklearn.utils.class_weight import compute_class_weight

# Calculate weights for 3D severity classes
# Severity has 2 classes, ensure we have weights for both
severity_classes_present_3d = np.unique(severity_classes_3d)
severity_weights_computed_3d = compute_class_weight(
    'balanced',
    classes=severity_classes_present_3d,
    y=severity_classes_3d
)

# Create full weight tensor for all severity classes (2 classes)
num_severity_classes_3d = len(severity_inverse_mapping_3d)
severity_class_weights_3d = torch.ones(num_severity_classes_3d, dtype=torch.float32)
for idx, class_idx in enumerate(severity_classes_present_3d):
    severity_class_weights_3d[class_idx] = severity_weights_computed_3d[idx]

# Calculate weights for 3D comparison score classes
# Comparison has 10 classes (0-9 representing scores 1-10)
comparison_classes_present_3d = np.unique(comparison_classes_3d)
comparison_weights_computed_3d = compute_class_weight(
    'balanced',
    classes=comparison_classes_present_3d,
    y=comparison_classes_3d
)

# Create full weight tensor for all 10 comparison classes
comparison_class_weights_3d = torch.ones(10, dtype=torch.float32)
for idx, class_idx in enumerate(comparison_classes_present_3d):
    comparison_class_weights_3d[class_idx] = comparison_weights_computed_3d[idx]

print("\n" + "="*70)
print("3D MODEL - CLASS IMBALANCE HANDLING")
print("="*70)
print(f"3D Severity class distribution: {np.bincount(severity_classes_3d)}")
print(f"3D Severity classes present: {severity_classes_present_3d}")
print(f"3D Severity class weights (all {num_severity_classes_3d} classes): {severity_class_weights_3d.numpy()}")
print(f"\n3D Comparison score class distribution: {np.bincount(comparison_classes_3d)}")
print(f"3D Comparison classes present: {comparison_classes_present_3d}")
print(f"3D Comparison class weights (all 10 classes): {comparison_class_weights_3d.numpy()}")
print("="*70)


In [ ]:
# ========================
# 3D Model - Step 4: Multi-Task ADOS Model (UPDATED ARCHITECTURE)
# ========================
# Note: We reuse the ImprovedADOSModel class defined earlier with updated architecture

sequence_input_size_3d = sequence_features_3d.shape[2]  # 222 features for 3D
demographic_input_size_3d = demographic_features_3d.shape[1]  # 2 (age, gender)

model_3d = ImprovedADOSModel(
    sequence_input_size=sequence_input_size_3d,
    demographic_input_size=demographic_input_size_3d,
    hidden_size=128,
    num_layers=2,
    dropout=0.4,
    num_outputs=4
)

print("3D Multi-Task Model Architecture (UPDATED):")
print(f"  Input: Sequence features ({sequence_input_size_3d} per frame) + Demographic features (age, gender)")
print("  Shared: Bidirectional LSTM + Attention + Demographic FC + Combined FC")
print("  Task-specific heads (MIXED):")
print("    - Severity: CLASSIFICATION (2 classes: severity 2 or 3)")
print("    - Social Affect: REGRESSION")
print("    - RRB: REGRESSION")
print("    - Comparison Score: CLASSIFICATION (10 classes: scores 1-10)")
print("\n✅ Same mixed architecture as 2D model, only input dimension changed (222 vs 150)")
print(f"Total parameters: {sum(p.numel() for p in model_3d.parameters()):,}")

In [ ]:
# ========================
# 3D Model - Step 5: Multi-Task Training (UPDATED FOR MIXED TASKS)
# ========================

# Optimizer with weight decay
optimizer_3d = torch.optim.AdamW(model_3d.parameters(), lr=1e-3, weight_decay=1e-4)

# Learning rate scheduler
scheduler_3d = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_3d, mode='min', factor=0.5, patience=5, verbose=True
)

# Early stopping
best_val_loss_3d = float('inf')
patience_3d = 15
patience_counter_3d = 0

num_epochs_3d = 100
# 🛡️ 3D MODEL ADVERSARIAL TRAINING CONFIGURATION
USE_ADVERSARIAL_TRAINING_3D = True  # Set to False to disable adversarial training
ADVERSARIAL_RATIO_3D = 0.3  # 30% of each batch will be adversarial examples
ATTACK_TYPE_3D = 'fgsm'  # 'fgsm' or 'pgd'
print("\n" + "="*70)
print("STARTING 3D MULTI-TASK TRAINING (2 CLASSIFICATION + 2 REGRESSION)")
print("="*70)
print(f"Epochs: {num_epochs_3d}")
print(f"Learning rate: 1e-3")
print(f"Early stopping patience: {patience_3d}")
print(f"📊 Class Imbalance: HANDLED with class weights")
print(f"🔢 Ordinal-Aware Loss: ENABLED for Comparison Score (EMD)")
print(f"🛡️ Adversarial Training: {'ENABLED' if USE_ADVERSARIAL_TRAINING_3D else 'DISABLED'}")
if USE_ADVERSARIAL_TRAINING_3D:
    print(f"   Attack Type: {ATTACK_TYPE_3D.upper()}")
    print(f"   Adversarial Ratio: {ADVERSARIAL_RATIO_3D*100:.0f}%")
print("Task types:")
print("  - Severity: CLASSIFICATION (2 classes)")
print("  - Social Affect: REGRESSION")
print("  - RRB: REGRESSION")
print("  - Comparison Score: ORDINAL CLASSIFICATION (10 classes with EMD loss)")
print("="*70 + "\n")

for epoch in range(num_epochs_3d):
    # Training
    model_3d.train()
    train_loss_3d = 0
    train_losses_sum_3d = {k: 0.0 for k in ['severity', 'social_affect', 'rrb', 'comparison']}
    
    for batch in train_loader_3d:
        x_seq, x_demo, y_sev_class, y_sa_reg, y_rrb_reg, y_comp_class, y_orig, lens = batch
        
        if USE_ADVERSARIAL_TRAINING_3D:
            # Use adversarial training step with 3D class weights
            loss, losses_dict = adversarial_training_step(
                model_3d, optimizer_3d, x_seq, x_demo, lens,
                y_sev_class, y_sa_reg, y_rrb_reg, y_comp_class,
                severity_weights=severity_class_weights_3d,
                comparison_weights=comparison_class_weights_3d,
                adv_ratio=ADVERSARIAL_RATIO_3D, attack_type=ATTACK_TYPE_3D
            )
        else:
            # Standard training step with 3D class weights
            optimizer_3d.zero_grad()
            out = model_3d(x_seq, x_demo, lens)
            loss, losses_dict = multi_task_loss(out, y_sev_class, y_sa_reg, y_rrb_reg, y_comp_class,
                                                 severity_weights=severity_class_weights_3d,
                                                 comparison_weights=comparison_class_weights_3d)
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model_3d.parameters(), max_norm=1.0)
            
            optimizer_3d.step()
        
        train_loss_3d += loss.item() * x_seq.size(0)
        for k in train_losses_sum_3d.keys():
            train_losses_sum_3d[k] += losses_dict[k].item() * x_seq.size(0)
    
    train_loss_3d /= len(train_loader_3d.dataset)
    for k in train_losses_sum_3d.keys():
        train_losses_sum_3d[k] /= len(train_loader_3d.dataset)
    
    # Validation (with 3D class weights for consistent loss calculation)
    model_3d.eval()
    val_loss_3d = 0
    val_losses_sum_3d = {k: 0.0 for k in ['severity', 'social_affect', 'rrb', 'comparison']}
    
    with torch.no_grad():
        for batch in val_loader_3d:
            x_seq, x_demo, y_sev_class, y_sa_reg, y_rrb_reg, y_comp_class, y_orig, lens = batch
            
            out = model_3d(x_seq, x_demo, lens)
            loss, losses_dict = multi_task_loss(out, y_sev_class, y_sa_reg, y_rrb_reg, y_comp_class,
                                                 severity_weights=severity_class_weights_3d,
                                                 comparison_weights=comparison_class_weights_3d)
            
            val_loss_3d += loss.item() * x_seq.size(0)
            for k in val_losses_sum_3d.keys():
                val_losses_sum_3d[k] += losses_dict[k].item() * x_seq.size(0)
    
    val_loss_3d /= len(val_loader_3d.dataset)
    for k in val_losses_sum_3d.keys():
        val_losses_sum_3d[k] /= len(val_loader_3d.dataset)
    
    # Learning rate scheduling
    scheduler_3d.step(val_loss_3d)
    
    print(f"Epoch {epoch+1}/{num_epochs_3d}")
    print(f"  Train Loss: {train_loss_3d:.4f} | Val Loss: {val_loss_3d:.4f}")
    print(f"  Task Losses (Val):")
    print(f"    Severity (CE): {val_losses_sum_3d['severity']:.4f}")
    print(f"    Social Affect (Huber): {val_losses_sum_3d['social_affect']:.4f}")
    print(f"    RRB (Huber): {val_losses_sum_3d['rrb']:.4f}")
    print(f"    Comparison (CE): {val_losses_sum_3d['comparison']:.4f}")
    
    # Early stopping
    if val_loss_3d < best_val_loss_3d:
        best_val_loss_3d = val_loss_3d
        patience_counter_3d = 0
        torch.save(model_3d.state_dict(), 'best_ados_multitask_model_3d.pth')
    else:
        patience_counter_3d += 1
        if patience_counter_3d >= patience_3d:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

# Load best model
model_3d.load_state_dict(torch.load('best_ados_multitask_model_3d.pth'))
print("\n✅ Best 3D model loaded")
print("📊 3D Model trained with class weights to handle imbalance")


In [ ]:
# ========================
# 3D Model - Step 6: Comprehensive Evaluation Metrics (UPDATED FOR MIXED TASKS)
# ========================
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

model_3d.eval()

# Collect predictions and targets
all_severity_preds_3d = []
all_severity_targets_3d = []
all_social_affect_preds_3d = []
all_social_affect_targets_3d = []
all_rrb_preds_3d = []
all_rrb_targets_3d = []
all_comparison_preds_3d = []
all_comparison_targets_3d = []
all_original_labels_3d = []

with torch.no_grad():
    for batch in val_loader_3d:
        x_seq, x_demo, y_sev_class, y_sa_reg, y_rrb_reg, y_comp_class, y_orig, lens = batch
        
        out = model_3d(x_seq, x_demo, lens)
        
        # Get class predictions (argmax for classification)
        severity_pred_class = torch.argmax(out['severity_logits'], dim=1)
        comparison_pred_class = torch.argmax(out['comparison_logits'], dim=1)
        
        all_severity_preds_3d.append(severity_pred_class.numpy())
        all_severity_targets_3d.append(y_sev_class.numpy())
        
        all_social_affect_preds_3d.append(out['social_affect'].squeeze().numpy())
        all_social_affect_targets_3d.append(y_sa_reg.numpy())
        
        all_rrb_preds_3d.append(out['rrb'].squeeze().numpy())
        all_rrb_targets_3d.append(y_rrb_reg.numpy())
        
        all_comparison_preds_3d.append(comparison_pred_class.numpy())
        all_comparison_targets_3d.append(y_comp_class.numpy())
        
        all_original_labels_3d.append(y_orig.numpy())

# Concatenate all batches
severity_pred_classes_3d = np.concatenate(all_severity_preds_3d)
severity_true_classes_3d = np.concatenate(all_severity_targets_3d)

social_affect_pred_scaled_3d = np.concatenate(all_social_affect_preds_3d)
social_affect_true_scaled_3d = np.concatenate(all_social_affect_targets_3d)

rrb_pred_scaled_3d = np.concatenate(all_rrb_preds_3d)
rrb_true_scaled_3d = np.concatenate(all_rrb_targets_3d)

comparison_pred_classes_3d = np.concatenate(all_comparison_preds_3d)
comparison_true_classes_3d = np.concatenate(all_comparison_targets_3d)

original_labels_3d = np.concatenate(all_original_labels_3d)

# Convert predictions back to original values
# Severity: map class indices back to original values
severity_pred_original_3d = np.array([severity_inverse_mapping_3d[cls] for cls in severity_pred_classes_3d])
severity_true_original_3d = original_labels_3d[:, 0]

# Social Affect and RRB: inverse transform
social_affect_pred_original_3d = social_affect_scaler_3d.inverse_transform(social_affect_pred_scaled_3d.reshape(-1, 1)).flatten()
social_affect_true_original_3d = original_labels_3d[:, 1]

rrb_pred_original_3d = rrb_scaler_3d.inverse_transform(rrb_pred_scaled_3d.reshape(-1, 1)).flatten()
rrb_true_original_3d = original_labels_3d[:, 2]

# Comparison Score: add 1 to convert from 0-9 to 1-10
comparison_pred_original_3d = comparison_pred_classes_3d + 1
comparison_true_original_3d = original_labels_3d[:, 3].astype(int)

print("\n" + "="*80)
print("3D MODEL - FINAL VALIDATION METRICS - MIXED CLASSIFICATION & REGRESSION")
print("="*80)
print("Using Age and Gender as INPUT features")
print("="*80)

# ===== SEVERITY (CLASSIFICATION) =====
print("\n" + "-"*80)
print("SEVERITY - CLASSIFICATION (2 classes)")
print("-"*80)
acc_severity_3d = accuracy_score(severity_true_classes_3d, severity_pred_classes_3d)
print(f"Accuracy: {acc_severity_3d:.4f}")

# Precision, Recall, F1
prec_3d, rec_3d, f1_sev_3d, support_3d = precision_recall_fscore_support(
    severity_true_classes_3d, severity_pred_classes_3d, average='weighted'
)
print(f"Precision (weighted): {prec_3d:.4f}")
print(f"Recall (weighted): {rec_3d:.4f}")
print(f"F1-Score (weighted): {f1_sev_3d:.4f}")

# Confusion Matrix
cm_severity_3d = confusion_matrix(severity_true_classes_3d, severity_pred_classes_3d)
print("\nConfusion Matrix:")
print(cm_severity_3d)

# Classification Report
print("\nDetailed Classification Report:")
severity_class_names_3d = [f"Severity {severity_inverse_mapping_3d[i]}" for i in range(len(severity_inverse_mapping_3d))]
print(classification_report(severity_true_classes_3d, severity_pred_classes_3d, target_names=severity_class_names_3d))

# ===== SOCIAL AFFECT (REGRESSION) =====
print("\n" + "-"*80)
print("SOCIAL AFFECT - REGRESSION")
print("-"*80)
mae_sa_3d = mean_absolute_error(social_affect_true_original_3d, social_affect_pred_original_3d)
rmse_sa_3d = np.sqrt(mean_squared_error(social_affect_true_original_3d, social_affect_pred_original_3d))
r2_sa_3d = r2_score(social_affect_true_original_3d, social_affect_pred_original_3d)
relative_mae_sa_3d = (mae_sa_3d / np.mean(social_affect_true_original_3d)) * 100 if np.mean(social_affect_true_original_3d) != 0 else 0

print(f"MAE:  {mae_sa_3d:.4f}")
print(f"RMSE: {rmse_sa_3d:.4f}")
print(f"R²:   {r2_sa_3d:.4f}")
print(f"Relative MAE: {relative_mae_sa_3d:.2f}%")
print(f"Mean True Value: {np.mean(social_affect_true_original_3d):.2f}")

# ===== RRB (REGRESSION) =====
print("\n" + "-"*80)
print("RRB - REGRESSION")
print("-"*80)
mae_rrb_3d = mean_absolute_error(rrb_true_original_3d, rrb_pred_original_3d)
rmse_rrb_3d = np.sqrt(mean_squared_error(rrb_true_original_3d, rrb_pred_original_3d))
r2_rrb_3d = r2_score(rrb_true_original_3d, rrb_pred_original_3d)
relative_mae_rrb_3d = (mae_rrb_3d / np.mean(rrb_true_original_3d)) * 100 if np.mean(rrb_true_original_3d) != 0 else 0

print(f"MAE:  {mae_rrb_3d:.4f}")
print(f"RMSE: {rmse_rrb_3d:.4f}")
print(f"R²:   {r2_rrb_3d:.4f}")
print(f"Relative MAE: {relative_mae_rrb_3d:.2f}%")
print(f"Mean True Value: {np.mean(rrb_true_original_3d):.2f}")

# ===== COMPARISON SCORE (CLASSIFICATION) =====
print("\n" + "-"*80)
print("COMPARISON SCORE - CLASSIFICATION (10 classes: 1-10)")
print("-"*80)
acc_comparison_3d = accuracy_score(comparison_true_classes_3d, comparison_pred_classes_3d)
print(f"Accuracy: {acc_comparison_3d:.4f}")

# Precision, Recall, F1
prec_comp_3d, rec_comp_3d, f1_comp_3d, support_comp_3d = precision_recall_fscore_support(
    comparison_true_classes_3d, comparison_pred_classes_3d, average='weighted', zero_division=0
)
print(f"Precision (weighted): {prec_comp_3d:.4f}")
print(f"Recall (weighted): {rec_comp_3d:.4f}")
print(f"F1-Score (weighted): {f1_comp_3d:.4f}")

# MAE for comparison (treating as ordinal)
mae_comparison_3d = mean_absolute_error(comparison_true_original_3d, comparison_pred_original_3d)
print(f"MAE (ordinal): {mae_comparison_3d:.4f}")

# Confusion Matrix

# Confusion Matrix
# Get unique classes that actually appear in the data
unique_comparison_classes_3d = np.unique(np.concatenate([comparison_true_classes_3d, comparison_pred_classes_3d]))
cm_comparison = confusion_matrix(comparison_true_classes_3d, comparison_pred_classes, labels=unique_comparison_classes_3d)
print("\nConfusion Matrix:")
print(cm_comparison)

# Classification Report
print("\nDetailed Classification Report:")
# Only create names for classes that actually exist
comparison_class_names_3d = [f"Score {i+1}" for i in unique_comparison_classes_3d]
print(classification_report(comparison_true_classes_3d, comparison_pred_classes_3d, 
                          labels=unique_comparison_classes_3d,
                          target_names=comparison_class_names_3d, zero_division=0))
cm_comparison_3d = confusion_matrix(comparison_true_classes_3d, comparison_pred_classes_3d)
print("\nConfusion Matrix:")
print(cm_comparison_3d)


print("\n" + "="*80)
print("3D MODEL - OVERALL PERFORMANCE SUMMARY:")
print("-" * 80)
print(f"Classification Tasks:")
print(f"  Severity - Accuracy: {acc_severity_3d:.4f}, F1: {f1_sev_3d:.4f}")
print(f"  Comparison Score - Accuracy: {acc_comparison_3d:.4f}, F1: {f1_comp_3d:.4f}, MAE: {mae_comparison_3d:.4f}")
print(f"\nRegression Tasks:")
print(f"  Social Affect - R²: {r2_sa_3d:.4f}, MAE: {mae_sa_3d:.4f}")
print(f"  RRB - R²: {r2_rrb_3d:.4f}, MAE: {mae_rrb_3d:.4f}")
print("="*80)

# Save predictions
results_df_3d = pd.DataFrame({
    'Severity_True': severity_true_original_3d,
    'Severity_Pred': severity_pred_original_3d,
    'SocialAffect_True': social_affect_true_original_3d,
    'SocialAffect_Pred': social_affect_pred_original_3d,
    'RRB_True': rrb_true_original_3d,
    'RRB_Pred': rrb_pred_original_3d,
    'ComparisonScore_True': comparison_true_original_3d,
    'ComparisonScore_Pred': comparison_pred_original_3d
})

results_df_3d.to_csv('multitask_predictions_3d.csv', index=False)
print("\n✅ 3D predictions saved to 'multitask_predictions_3d.csv'")

In [ ]:
# ========================
# 3D Model - Step 7: Visualization of Results (UPDATED FOR MIXED TASKS)
# ========================
import matplotlib.pyplot as plt
import seaborn as sns

fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

fig.suptitle('3D Multi-Task ADOS Prediction Results (Mixed Classification & Regression)', 
             fontsize=16, fontweight='bold')

# ===== SEVERITY (CLASSIFICATION - Confusion Matrix) =====
ax1 = fig.add_subplot(gs[0, 0])
sns.heatmap(cm_severity_3d, annot=True, fmt='d', cmap='Blues', ax=ax1, cbar=True)
ax1.set_xlabel('Predicted Class')
ax1.set_ylabel('True Class')
ax1.set_title(f'Severity (Classification - 3D)\nAccuracy: {acc_severity_3d:.3f}, F1: {f1_sev_3d:.3f}')
ax1.set_xticklabels([f'Sev {severity_inverse_mapping_3d[i]}' for i in range(len(severity_inverse_mapping_3d))])
ax1.set_yticklabels([f'Sev {severity_inverse_mapping_3d[i]}' for i in range(len(severity_inverse_mapping_3d))])

# ===== SOCIAL AFFECT (REGRESSION - Scatter Plot) =====
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(social_affect_true_original_3d, social_affect_pred_original_3d, alpha=0.6, s=50, color='green')
min_val = min(social_affect_true_original_3d.min(), social_affect_pred_original_3d.min())
max_val = max(social_affect_true_original_3d.max(), social_affect_pred_original_3d.max())
ax2.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
ax2.set_xlabel('True Values', fontsize=11)
ax2.set_ylabel('Predicted Values', fontsize=11)
ax2.set_title(f'Social Affect (Regression - 3D)\nMAE: {mae_sa_3d:.3f}, R²: {r2_sa_3d:.3f}', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

# ===== RRB (REGRESSION - Scatter Plot) =====
ax3 = fig.add_subplot(gs[1, 0])
ax3.scatter(rrb_true_original_3d, rrb_pred_original_3d, alpha=0.6, s=50, color='green')
min_val = min(rrb_true_original_3d.min(), rrb_pred_original_3d.min())
max_val = max(rrb_true_original_3d.max(), rrb_pred_original_3d.max())
ax3.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
ax3.set_xlabel('True Values', fontsize=11)
ax3.set_ylabel('Predicted Values', fontsize=11)
ax3.set_title(f'RRB (Regression - 3D)\nMAE: {mae_rrb_3d:.3f}, R²: {r2_rrb_3d:.3f}', fontsize=12)
ax3.legend()
ax3.grid(True, alpha=0.3)

# ===== COMPARISON SCORE (CLASSIFICATION - Confusion Matrix) =====
ax4 = fig.add_subplot(gs[1, 1])
# Show only non-zero classes in confusion matrix for clarity
unique_classes_3d = np.unique(np.concatenate([comparison_true_classes_3d, comparison_pred_classes_3d]))
cm_comparison_filtered_3d = confusion_matrix(comparison_true_classes_3d, comparison_pred_classes_3d, labels=unique_classes_3d)
sns.heatmap(cm_comparison_filtered_3d, annot=True, fmt='d', cmap='Greens', ax=ax4, cbar=True)
ax4.set_xlabel('Predicted Score')
ax4.set_ylabel('True Score')
ax4.set_title(f'Comparison Score (Classification - 3D)\nAccuracy: {acc_comparison_3d:.3f}, F1: {f1_comp_3d:.3f}, MAE: {mae_comparison_3d:.3f}')
ax4.set_xticklabels([f'{i+1}' for i in unique_classes_3d], rotation=0)
ax4.set_yticklabels([f'{i+1}' for i in unique_classes_3d], rotation=0)

# ===== COMPARISON SCORE (Ordinal Error Distribution) =====
ax5 = fig.add_subplot(gs[2, :])
errors_3d = comparison_pred_original_3d - comparison_true_original_3d
error_bins = np.arange(-9.5, 10.5, 1)
ax5.hist(errors_3d, bins=error_bins, color='purple', alpha=0.7, edgecolor='black')
ax5.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction')
ax5.set_xlabel('Prediction Error (Predicted - True)', fontsize=11)
ax5.set_ylabel('Frequency', fontsize=11)
ax5.set_title('3D Comparison Score - Prediction Error Distribution', fontsize=12)
ax5.legend()
ax5.grid(True, alpha=0.3, axis='y')

plt.savefig('multitask_results_3d.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 3D visualization saved to 'multitask_results_3d.png'")

In [ ]:
# ========================
# 3D Model - Step 8: Multi-Task Explainability (Same as 2D)
# ========================
# The explainer can be reused with the 3D model

def explain_multitask_samples_3d(model, val_loader, target_scalers, demographic_scaler, 
                                  fps=30, num_samples=3, device='cpu'):
    """Generate explanations for 3D multi-task model"""
    explainer = MultiTaskADOSExplainer(
        model=model,
        target_scalers=target_scalers,
        demographic_scaler=demographic_scaler,
        fps=fps,
        device=device
    )
    
    explanations = []
    
    print(f"\n{'='*70}")
    print(f"Generating 3D Explanations for {num_samples} samples...")
    print(f"{'='*70}\n")
    
    sample_count = 0
    for batch in val_loader:
        x_seq, x_demo, y_sev_class, y_sa_reg, y_rrb_reg, y_comp_class, y_orig, lens = batch
        
        for i in range(x_seq.shape[0]):
            if sample_count >= num_samples:
                break
            
            seq = x_seq[i]
            demo = x_demo[i]
            y_true_orig = y_orig[i].numpy()
            seq_len = lens[i]
            
            # Get demographics in original scale
            demo_orig = demographic_scaler.inverse_transform(demo.numpy().reshape(1, -1))[0]
            
            print(f"\n{'='*70}")
            print(f"3D SAMPLE {sample_count + 1}/{num_samples}")
            print(f"Age: {demo_orig[0]:.1f}, Gender: {int(demo_orig[1])}")
            print(f"{'='*70}")
            
            # Generate explanation
            explanation = explainer.explain_all_tasks(
                x_seq=seq,
                x_demo=demo,
                seq_len=seq_len
            )
            
            # Add ground truth
            explanation['ground_truth'] = {
                'severity': float(y_true_orig[0]),
                'social_affect': float(y_true_orig[1]),
                'rrb': float(y_true_orig[2]),
                'comparison_score': float(y_true_orig[3])
            }
            
            # Print summary
            print(explanation['summary'])
            
            print("\nGround Truth vs Predictions:")
            for j, task in enumerate(['severity', 'social_affect', 'rrb', 'comparison_score']):
                true_val = explanation['ground_truth'][task]
                pred_val = explanation['predictions'][task]
                error = abs(true_val - pred_val)
                print(f"  {task}: True={true_val:.2f}, Pred={pred_val:.2f}, Error={error:.2f}")
            
            # Print temporal insights
            print("\n" + "="*80)
            print("3D TEMPORAL INSIGHTS (Critical Time Segments):")
            print("="*80)
            for task_name in ['Severity', 'Social Affect', 'RRB']:
                task_exp = explanation['task_explanations'][task_name]
                print(f"\n{task_name}:")
                
                print("  Problematic segments (increase score):")
                for seg in task_exp['temporal_segments']['positive_segments'][:3]:
                    print(f"    {seg['start_time']:.1f}-{seg['end_time']:.1f}s: contribution={seg['contribution']:.2f}")
                
                print("  Strength segments (decrease score):")
                for seg in task_exp['temporal_segments']['negative_segments'][:2]:
                    print(f"    {seg['start_time']:.1f}-{seg['end_time']:.1f}s: contribution={seg['contribution']:.2f}")
            
            explanations.append(explanation)
            
            # Visualize
            explainer.visualize_explanation(
                explanation, 
                save_path=f'multitask_explanation_3d_{sample_count+1}.png'
            )
            
            sample_count += 1
        
        if sample_count >= num_samples:
            break
    
    # Save for therapy engine
    if explanations:
        with open('multitask_therapy_input_3d.json', 'w') as f:
            json.dump(explanations[0], f, indent=2, default=str)
        print(f"\n✅ Saved 3D therapy engine input to multitask_therapy_input_3d.json")
    
    return explanations

# Run the 3D explainer
explanations_3d = explain_multitask_samples_3d(
    model=model_3d,
    val_loader=val_loader_3d,
    target_scalers=target_scalers_3d,
    demographic_scaler=demographic_scaler_3d,
    fps=30,
    num_samples=3,
    device='cpu'
)

print("\n" + "="*70)
print("3D MULTI-TASK EXPLAINABILITY COMPLETE!")
print("="*70)
print(f"\n✅ Generated {len(explanations_3d)} complete 3D explanations")
print("✅ Each explanation shows:")
print("   - Predictions for 4 ADOS metrics from 3D skeleton data")
print("   - How age and gender influence each prediction")
print("   - Which body joints contribute most to each score")
print("   - Temporal patterns and behavioral indicators")

In [ ]:
# ========================
# 3D Model - Step 9: Helper Functions for Inference (UPDATED FOR MIXED TASKS)
# ========================
def predict_single_sample_3d(model, sequence_folder_path, age, gender_value, 
                              feature_scaler, demographic_scaler, target_scalers, 
                              gender_encoder, severity_inverse_mapping, max_len=100):
    """
    Make prediction on a single 3D video sequence with demographic information (UPDATED)
    
    Args:
        model: Trained 3D multi-task model
        sequence_folder_path: Path to folder containing .npz frame files (3D)
        age: Patient's age (years)
        gender_value: Patient's gender (original value, e.g., 'M', 'F')
        feature_scaler: Fitted StandardScaler for 3D skeletal features
        demographic_scaler: Fitted StandardScaler for demographics
        target_scalers: List of scalers [None, social_affect_scaler, rrb_scaler, None]
        gender_encoder: LabelEncoder for gender
        severity_inverse_mapping: Dict to map class indices back to severity values
        max_len: Maximum sequence length
    
    Returns:
        Dictionary with all predictions in original scale
    """
    # Load and process 3D sequence
    sequence = load_npz_sequence_3d(sequence_folder_path)
    
    if sequence.shape[0] == 0:
        return {"error": "Empty sequence"}
    
    # Preprocess 3D skeletal features
    original_length = min(sequence.shape[0], max_len)
    sequence = normalize_skeleton_3d(sequence)
    sequence = pad_sequence_3d(sequence, max_len)
    features = compute_enhanced_features_3d(sequence)
    
    # Normalize skeletal features
    features_flat = features.reshape(-1, features.shape[-1])
    features_normalized = feature_scaler.transform(features_flat)
    features_normalized = features_normalized.reshape(features.shape)
    
    # Encode and normalize demographic features
    gender_encoded = gender_encoder.transform([gender_value])[0]
    demo_features = np.array([[age, gender_encoded]])
    demo_normalized = demographic_scaler.transform(demo_features)
    
    # Convert to tensors
    x_seq = torch.tensor(features_normalized, dtype=torch.float32).unsqueeze(0)
    x_demo = torch.tensor(demo_normalized, dtype=torch.float32).squeeze(0)
    seq_len = torch.tensor([original_length], dtype=torch.long)
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(x_seq, x_demo, seq_len)
    
    # Extract predictions
    # Severity: classification - get predicted class
    severity_class = torch.argmax(outputs['severity_logits'], dim=1).item()
    severity_value = severity_inverse_mapping[severity_class]
    
    # Social Affect: regression - inverse transform
    social_affect_scaled = outputs['social_affect'].squeeze().item()
    social_affect_value = target_scalers[1].inverse_transform([[social_affect_scaled]])[0, 0]
    
    # RRB: regression - inverse transform
    rrb_scaled = outputs['rrb'].squeeze().item()
    rrb_value = target_scalers[2].inverse_transform([[rrb_scaled]])[0, 0]
    
    # Comparison Score: classification - get predicted class (0-9) and add 1 to get (1-10)
    comparison_class = torch.argmax(outputs['comparison_logits'], dim=1).item()
    comparison_value = comparison_class + 1
    
    predictions = {
        'severity': float(severity_value),
        'social_affect': float(social_affect_value),
        'rrb': float(rrb_value),
        'comparison_score': int(comparison_value),
        'input_age': age,
        'input_gender': gender_value
    }
    
    return predictions

print("✅ 3D helper functions for inference ready (UPDATED FOR MIXED TASKS)")
print("\nUsage:")
print("  predict_single_sample_3d(model_3d, path, age, gender, ..., severity_inverse_mapping_3d) - Predict on one 3D video")
print("\nExample:")
print("  preds = predict_single_sample_3d(model_3d, 'path/to/video', age=8, gender_value='M', ...)")
print("  # Returns: {'severity': 2 or 3, 'social_affect': 8.5, 'rrb': 3.8, 'comparison_score': 7 (1-10)}")

In [ ]:
# ========================
# Model Comparison: 2D vs 3D (UPDATED FOR MIXED TASKS)
# ========================
import matplotlib.pyplot as plt
import pandas as pd

print("\n" + "="*80)
print("COMPARISON: 2D MODEL vs 3D MODEL (MIXED CLASSIFICATION & REGRESSION)")
print("="*80)

# Compute metrics for both models
comparison_data = []

# ===== SEVERITY (CLASSIFICATION) =====
task_name = 'Severity (Classification)'
comparison_data.append({
    'Task': task_name,
    'Type': 'Classification',
    '2D_Accuracy': acc_severity,
    '2D_F1': f1,
    '3D_Accuracy': acc_severity_3d,
    '3D_F1': f1_sev_3d,
    'Accuracy_Improvement': ((acc_severity_3d - acc_severity) / acc_severity) * 100 if acc_severity != 0 else 0,
    'F1_Improvement': ((f1_sev_3d - f1) / f1) * 100 if f1 != 0 else 0
})

# ===== SOCIAL AFFECT (REGRESSION) =====
task_name = 'Social Affect (Regression)'
comparison_data.append({
    'Task': task_name,
    'Type': 'Regression',
    '2D_MAE': mae_sa,
    '2D_RMSE': rmse_sa,
    '2D_R2': r2_sa,
    '3D_MAE': mae_sa_3d,
    '3D_RMSE': rmse_sa_3d,
    '3D_R2': r2_sa_3d,
    'MAE_Improvement': ((mae_sa - mae_sa_3d) / mae_sa) * 100,
    'RMSE_Improvement': ((rmse_sa - rmse_sa_3d) / rmse_sa) * 100,
    'R2_Improvement': ((r2_sa_3d - r2_sa) / abs(r2_sa)) * 100 if r2_sa != 0 else 0
})

# ===== RRB (REGRESSION) =====
task_name = 'RRB (Regression)'
comparison_data.append({
    'Task': task_name,
    'Type': 'Regression',
    '2D_MAE': mae_rrb,
    '2D_RMSE': rmse_rrb,
    '2D_R2': r2_rrb,
    '3D_MAE': mae_rrb_3d,
    '3D_RMSE': rmse_rrb_3d,
    '3D_R2': r2_rrb_3d,
    'MAE_Improvement': ((mae_rrb - mae_rrb_3d) / mae_rrb) * 100,
    'RMSE_Improvement': ((rmse_rrb - rmse_rrb_3d) / rmse_rrb) * 100,
    'R2_Improvement': ((r2_rrb_3d - r2_rrb) / abs(r2_rrb)) * 100 if r2_rrb != 0 else 0
})

# ===== COMPARISON SCORE (CLASSIFICATION) =====
task_name = 'Comparison Score (Classification)'
comparison_data.append({
    'Task': task_name,
    'Type': 'Classification',
    '2D_Accuracy': acc_comparison,
    '2D_F1': f1_comp,
    '2D_MAE': mae_comparison,
    '3D_Accuracy': acc_comparison_3d,
    '3D_F1': f1_comp_3d,
    '3D_MAE': mae_comparison_3d,
    'Accuracy_Improvement': ((acc_comparison_3d - acc_comparison) / acc_comparison) * 100 if acc_comparison != 0 else 0,
    'F1_Improvement': ((f1_comp_3d - f1_comp) / f1_comp) * 100 if f1_comp != 0 else 0,
    'MAE_Improvement': ((mae_comparison - mae_comparison_3d) / mae_comparison) * 100
})

comparison_df = pd.DataFrame(comparison_data)

print("\nDetailed Comparison:")
print("-" * 80)

for _, row in comparison_df.iterrows():
    print(f"\n{row['Task']}:")
    if row['Type'] == 'Classification':
        print(f"  2D Model - Accuracy: {row.get('2D_Accuracy', 'N/A'):.4f}, F1: {row.get('2D_F1', 'N/A'):.4f}")
        print(f"  3D Model - Accuracy: {row.get('3D_Accuracy', 'N/A'):.4f}, F1: {row.get('3D_F1', 'N/A'):.4f}")
        print(f"  Improvement - Accuracy: {row.get('Accuracy_Improvement', 0):+.2f}%, F1: {row.get('F1_Improvement', 0):+.2f}%")
        if 'MAE' in row['Task']:
            print(f"  MAE (ordinal) - 2D: {row.get('2D_MAE', 'N/A'):.4f}, 3D: {row.get('3D_MAE', 'N/A'):.4f}, Improvement: {row.get('MAE_Improvement', 0):+.2f}%")
    else:  # Regression
        print(f"  2D Model - MAE: {row.get('2D_MAE', 'N/A'):.4f}, RMSE: {row.get('2D_RMSE', 'N/A'):.4f}, R²: {row.get('2D_R2', 'N/A'):.4f}")
        print(f"  3D Model - MAE: {row.get('3D_MAE', 'N/A'):.4f}, RMSE: {row.get('3D_RMSE', 'N/A'):.4f}, R²: {row.get('3D_R2', 'N/A'):.4f}")
        print(f"  Improvement - MAE: {row.get('MAE_Improvement', 0):+.2f}%, RMSE: {row.get('RMSE_Improvement', 0):+.2f}%, R²: {row.get('R2_Improvement', 0):+.2f}%")

# Overall comparison
print("\n" + "="*80)
print("OVERALL SUMMARY:")
print("-" * 80)
print("Classification Tasks:")
print(f"  Severity - 2D Accuracy: {acc_severity:.4f}, 3D Accuracy: {acc_severity_3d:.4f}")
print(f"  Comparison Score - 2D Accuracy: {acc_comparison:.4f}, 3D Accuracy: {acc_comparison_3d:.4f}")
print("\nRegression Tasks:")
print(f"  Social Affect - 2D R²: {r2_sa:.4f}, 3D R²: {r2_sa_3d:.4f}")
print(f"  RRB - 2D R²: {r2_rrb:.4f}, 3D R²: {r2_rrb_3d:.4f}")
print("="*80)

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('2D vs 3D Model Comparison (Mixed Classification & Regression)', fontsize=16, fontweight='bold')

# ===== Classification Metrics =====
# Severity Accuracy
ax1 = axes[0, 0]
tasks_class = ['Severity', 'Comparison Score']
acc_2d = [acc_severity, acc_comparison]
acc_3d = [acc_severity_3d, acc_comparison_3d]
x = np.arange(len(tasks_class))
width = 0.35
ax1.bar(x - width/2, acc_2d, width, label='2D', color='steelblue', alpha=0.8)
ax1.bar(x + width/2, acc_3d, width, label='3D', color='green', alpha=0.8)
ax1.set_ylabel('Accuracy')
ax1.set_title('Classification Accuracy')
ax1.set_xticks(x)
ax1.set_xticklabels(tasks_class)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# F1 Score
ax2 = axes[0, 1]
f1_2d = [f1, f1_comp]
f1_3d = [f1_sev_3d, f1_comp_3d]
ax2.bar(x - width/2, f1_2d, width, label='2D', color='steelblue', alpha=0.8)
ax2.bar(x + width/2, f1_3d, width, label='3D', color='green', alpha=0.8)
ax2.set_ylabel('F1 Score')
ax2.set_title('Classification F1 Score')
ax2.set_xticks(x)
ax2.set_xticklabels(tasks_class)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# MAE for Comparison Score
ax3 = axes[0, 2]
mae_2d = mae_comparison
mae_3d = mae_comparison_3d
ax3.bar(['2D', '3D'], [mae_2d, mae_3d], color=['steelblue', 'green'], alpha=0.8)
ax3.set_ylabel('MAE (ordinal)')
ax3.set_title('Comparison Score MAE')
ax3.grid(axis='y', alpha=0.3)

# ===== Regression Metrics =====
# MAE
ax4 = axes[1, 0]
tasks_reg = ['Social Affect', 'RRB']
mae_2d_reg = [mae_sa, mae_rrb]
mae_3d_reg = [mae_sa_3d, mae_rrb_3d]
x_reg = np.arange(len(tasks_reg))
ax4.bar(x_reg - width/2, mae_2d_reg, width, label='2D', color='steelblue', alpha=0.8)
ax4.bar(x_reg + width/2, mae_3d_reg, width, label='3D', color='green', alpha=0.8)
ax4.set_ylabel('MAE')
ax4.set_title('Regression MAE')
ax4.set_xticks(x_reg)
ax4.set_xticklabels(tasks_reg)
ax4.legend()
ax4.grid(axis='y', alpha=0.3)

# RMSE
ax5 = axes[1, 1]
rmse_2d_reg = [rmse_sa, rmse_rrb]
rmse_3d_reg = [rmse_sa_3d, rmse_rrb_3d]
ax5.bar(x_reg - width/2, rmse_2d_reg, width, label='2D', color='steelblue', alpha=0.8)
ax5.bar(x_reg + width/2, rmse_3d_reg, width, label='3D', color='green', alpha=0.8)
ax5.set_ylabel('RMSE')
ax5.set_title('Regression RMSE')
ax5.set_xticks(x_reg)
ax5.set_xticklabels(tasks_reg)
ax5.legend()
ax5.grid(axis='y', alpha=0.3)

# R² Score
ax6 = axes[1, 2]
r2_2d_reg = [r2_sa, r2_rrb]
r2_3d_reg = [r2_sa_3d, r2_rrb_3d]
ax6.bar(x_reg - width/2, r2_2d_reg, width, label='2D', color='steelblue', alpha=0.8)
ax6.bar(x_reg + width/2, r2_3d_reg, width, label='3D', color='green', alpha=0.8)
ax6.set_ylabel('R² Score')
ax6.set_title('Regression R² (Higher is Better)')
ax6.set_xticks(x_reg)
ax6.set_xticklabels(tasks_reg)
ax6.legend()
ax6.grid(axis='y', alpha=0.3)
ax6.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig('2d_vs_3d_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Comparison visualization saved to '2d_vs_3d_comparison.png'")

# Save comparison table
comparison_df.to_csv('2d_vs_3d_comparison.csv', index=False)
print("✅ Comparison table saved to '2d_vs_3d_comparison.csv'")

# Exporting models

In [10]:
# ========================
# EXPORT 2D MODEL FOR DEPLOYMENT
# ========================
import pickle
import json
from datetime import datetime

# Create export directory
export_dir_2d = '2d_model_export'
os.makedirs(export_dir_2d, exist_ok=True)

print("\n" + "="*80)
print("EXPORTING 2D MODEL FOR DEPLOYMENT")
print("="*80)

# 1. Save model state dict
model_path_2d = os.path.join(export_dir_2d, '2d_model.pth')
torch.save(model.state_dict(), model_path_2d)
print(f"✅ Model saved to: {model_path_2d}")

# 2. Save all scalers and encoders
scalers_2d = {
    'feature_scaler': feature_scaler,
    'demographic_scaler': demographic_scaler,
    'social_affect_scaler': target_scalers[1],
    'rrb_scaler': target_scalers[2],
    'gender_encoder': gender_encoder
}

scalers_path_2d = os.path.join(export_dir_2d, '2d_scalers.pkl')
with open(scalers_path_2d, 'wb') as f:
    pickle.dump(scalers_2d, f)
print(f"✅ Scalers saved to: {scalers_path_2d}")

# 3. Save severity mapping
mappings_2d = {
    'severity_inverse_mapping': severity_inverse_mapping,
    'severity_mapping': {int(k): int(v) for k, v in severity_mapping.items()},
    'gender_classes': gender_encoder.classes_.tolist()
}

mappings_path_2d = os.path.join(export_dir_2d, '2d_mappings.json')
with open(mappings_path_2d, 'w') as f:
    json.dump(mappings_2d, f, indent=2)
print(f"✅ Mappings saved to: {mappings_path_2d}")

# 4. Save model configuration
model_config_2d = {
    'model_type': '2D',
    'sequence_input_size': sequence_input_size,
    'demographic_input_size': demographic_input_size,
    'hidden_size': 128,
    'num_layers': 2,
    'dropout': 0.4,
    'num_outputs': 4,
    'max_sequence_length': 100,
    'task_types': {
        'severity': 'classification',
        'social_affect': 'regression',
        'rrb': 'regression',
        'comparison_score': 'classification'
    },
    'num_classes': {
        'severity': 2,
        'comparison_score': 10
    },
    'export_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'best_val_loss': float(best_val_loss)
}

config_path_2d = os.path.join(export_dir_2d, '2d_model_config.json')
with open(config_path_2d, 'w') as f:
    json.dump(model_config_2d, f, indent=2)
print(f"✅ Model config saved to: {config_path_2d}")

# 5. Save model performance metrics
performance_2d = {
    'severity': {
        'type': 'classification',
        'accuracy': float(acc_severity),
        'f1_score': float(f1),
        'precision': float(prec),
        'recall': float(rec)
    },
    'social_affect': {
        'type': 'regression',
        'mae': float(mae_sa),
        'rmse': float(rmse_sa),
        'r2': float(r2_sa)
    },
    'rrb': {
        'type': 'regression',
        'mae': float(mae_rrb),
        'rmse': float(rmse_rrb),
        'r2': float(r2_rrb)
    },
    'comparison_score': {
        'type': 'classification',
        'accuracy': float(acc_comparison),
        'f1_score': float(f1_comp),
        'mae_ordinal': float(mae_comparison)
    }
}

performance_path_2d = os.path.join(export_dir_2d, '2d_performance_metrics.json')
with open(performance_path_2d, 'w') as f:
    json.dump(performance_2d, f, indent=2)
print(f"✅ Performance metrics saved to: {performance_path_2d}")

print("\n" + "="*80)
print("2D MODEL EXPORT COMPLETE!")
print("="*80)
print(f"\nExported files in '{export_dir_2d}':")
print(f"  1. 2d_ados_model.pth - Model weights")
print(f"  2. 2d_scalers.pkl - All scalers and encoders")
print(f"  3. 2d_mappings.json - Severity and gender mappings")
print(f"  4. 2d_model_config.json - Model architecture configuration")
print(f"  5. 2d_performance_metrics.json - Model performance metrics")
print("\n" + "="*80)


EXPORTING 2D MODEL FOR DEPLOYMENT
✅ Model saved to: 2d_model_export/2d_model.pth
✅ Scalers saved to: 2d_model_export/2d_scalers.pkl
✅ Mappings saved to: 2d_model_export/2d_mappings.json
✅ Model config saved to: 2d_model_export/2d_model_config.json
✅ Performance metrics saved to: 2d_model_export/2d_performance_metrics.json

2D MODEL EXPORT COMPLETE!

Exported files in '2d_model_export':
  1. 2d_ados_model.pth - Model weights
  2. 2d_scalers.pkl - All scalers and encoders
  3. 2d_mappings.json - Severity and gender mappings
  4. 2d_model_config.json - Model architecture configuration
  5. 2d_performance_metrics.json - Model performance metrics



In [ ]:
# ========================
# EXPORT 3D MODEL FOR DEPLOYMENT
# ========================
import pickle
import json
from datetime import datetime

# Create export directory
export_dir_3d = '3d_model_export'
os.makedirs(export_dir_3d, exist_ok=True)

print("\n" + "="*80)
print("EXPORTING 3D MODEL FOR DEPLOYMENT")
print("="*80)

# 1. Save model state dict
model_path_3d = os.path.join(export_dir_3d, '3d_model.pth')
torch.save(model_3d.state_dict(), model_path_3d)
print(f"✅ Model saved to: {model_path_3d}")

# 2. Save all scalers and encoders
scalers_3d = {
    'feature_scaler': feature_scaler_3d,
    'demographic_scaler': demographic_scaler_3d,
    'social_affect_scaler': target_scalers_3d[1],
    'rrb_scaler': target_scalers_3d[2],
    'gender_encoder': gender_encoder_3d
}

scalers_path_3d = os.path.join(export_dir_3d, '3d_scalers.pkl')
with open(scalers_path_3d, 'wb') as f:
    pickle.dump(scalers_3d, f)
print(f"✅ Scalers saved to: {scalers_path_3d}")

# 3. Save severity mapping
mappings_3d = {
    'severity_inverse_mapping': severity_inverse_mapping_3d,
    'severity_mapping': {int(k): int(v) for k, v in severity_mapping_3d.items()},
    'gender_classes': gender_encoder_3d.classes_.tolist()
}

mappings_path_3d = os.path.join(export_dir_3d, '3d_mappings.json')
with open(mappings_path_3d, 'w') as f:
    json.dump(mappings_3d, f, indent=2)
print(f"✅ Mappings saved to: {mappings_path_3d}")

# 4. Save model configuration
model_config_3d = {
    'model_type': '3D',
    'sequence_input_size': sequence_input_size_3d,
    'demographic_input_size': demographic_input_size_3d,
    'hidden_size': 128,
    'num_layers': 2,
    'dropout': 0.4,
    'num_outputs': 4,
    'max_sequence_length': 100,
    'task_types': {
        'severity': 'classification',
        'social_affect': 'regression',
        'rrb': 'regression',
        'comparison_score': 'classification'
    },
    'num_classes': {
        'severity': 2,
        'comparison_score': 10
    },
    'export_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'best_val_loss': float(best_val_loss_3d)
}

config_path_3d = os.path.join(export_dir_3d, '3d_model_config.json')
with open(config_path_3d, 'w') as f:
    json.dump(model_config_3d, f, indent=2)
print(f"✅ Model config saved to: {config_path_3d}")

# 5. Save model performance metrics
performance_3d = {
    'severity': {
        'type': 'classification',
        'accuracy': float(acc_severity_3d),
        'f1_score': float(f1_sev_3d),
        'precision': float(prec_3d),
        'recall': float(rec_3d)
    },
    'social_affect': {
        'type': 'regression',
        'mae': float(mae_sa_3d),
        'rmse': float(rmse_sa_3d),
        'r2': float(r2_sa_3d)
    },
    'rrb': {
        'type': 'regression',
        'mae': float(mae_rrb_3d),
        'rmse': float(rmse_rrb_3d),
        'r2': float(r2_rrb_3d)
    },
    'comparison_score': {
        'type': 'classification',
        'accuracy': float(acc_comparison_3d),
        'f1_score': float(f1_comp_3d),
        'mae_ordinal': float(mae_comparison_3d)
    }
}

performance_path_3d = os.path.join(export_dir_3d, '3d_performance_metrics.json')
with open(performance_path_3d, 'w') as f:
    json.dump(performance_3d, f, indent=2)
print(f"✅ Performance metrics saved to: {performance_path_3d}")

print("\n" + "="*80)
print("3D MODEL EXPORT COMPLETE!")
print("="*80)
print(f"\nExported files in '{export_dir_3d}':")
print(f"  1. 3d_ados_model.pth - Model weights")
print(f"  2. 3d_scalers.pkl - All scalers and encoders")
print(f"  3. 3d_mappings.json - Severity and gender mappings")
print(f"  4. 3d_model_config.json - Model architecture configuration")
print(f"  5. 3d_performance_metrics.json - Model performance metrics")
print("\n" + "="*80)

In [ ]:
# ========================
# COMPLETE INFERENCE EXAMPLE FOR SERVER DEPLOYMENT (WITH EXPLAINABILITY + SECURITY)
# ========================

def defensive_preprocessing(sequence, smoothing_window=5):
    """
    🛡️ Apply defensive preprocessing to reduce adversarial perturbations
    
    Args:
        sequence: Input sequence [seq_len, num_joints, features]
        smoothing_window: Window size for temporal smoothing
    
    Returns:
        smoothed_sequence: Preprocessed sequence resistant to adversarial attacks
    """
    from scipy.signal import savgol_filter
    from scipy.ndimage import median_filter
    
    # Temporal smoothing (removes high-frequency adversarial noise)
    try:
        smoothed = savgol_filter(sequence, smoothing_window, 3, axis=0)
    except:
        # Fallback to simple moving average if savgol fails
        smoothed = sequence.copy()
        for i in range(smoothing_window//2, len(sequence) - smoothing_window//2):
            smoothed[i] = np.mean(sequence[i-smoothing_window//2:i+smoothing_window//2+1], axis=0)
    
    # Median filtering (robust to outliers and adversarial perturbations)
    smoothed = median_filter(smoothed, size=(3, 1, 1))
    
    return smoothed


def make_prediction_on_new_video(video_folder_path, age, gender, model_type='2D', 
                                include_explainability=True, apply_defensive_preprocessing=True):
    """
    Complete inference pipeline for a new video with explainability and security features
    
    Args:
        video_folder_path: Path to folder containing .npz frame files
        age: Patient age in years (e.g., 8.5)
        gender: Patient gender ('M' or 'F')
        model_type: '2D' or '3D'
        include_explainability: Whether to include explainability analysis (default: True)
        apply_defensive_preprocessing: 🛡️ Apply defensive preprocessing against adversarial attacks (default: True)
    
    Returns:
        Dictionary with predictions for all 4 ADOS metrics and explainability
    """
    import torch
    import numpy as np
    
    # Load the appropriate model
    if model_type == '2D':
        loaded = load_2d_model_for_inference('2d_model_export')
        load_func = load_npz_sequence
        normalize_func = normalize_skeleton
        pad_func = pad_sequence
        feature_func = compute_enhanced_features
        max_len = 100
    else:  # 3D
        loaded = load_3d_model_for_inference('3d_model_export')
        load_func = load_npz_sequence_3d
        normalize_func = normalize_skeleton_3d
        pad_func = pad_sequence_3d
        feature_func = compute_enhanced_features_3d
        max_len = 100
    
    model = loaded['model']
    scalers = loaded['scalers']
    mappings = loaded['mappings']
    
    # 1. Load and preprocess video
    sequence = load_func(video_folder_path)
    
    if sequence.shape[0] == 0:
        return {"error": "Empty sequence"}
    
    # 🛡️ 2. Apply defensive preprocessing (security feature)
    if apply_defensive_preprocessing:
        sequence = defensive_preprocessing(sequence, smoothing_window=5)
    
    # 3. Process skeletal features
    original_length = min(sequence.shape[0], max_len)
    sequence = normalize_func(sequence)
    sequence = pad_func(sequence, max_len)
    features = feature_func(sequence)
    
    # 3. Normalize skeletal features
    features_flat = features.reshape(-1, features.shape[-1])
    features_normalized = scalers['feature_scaler'].transform(features_flat)
    features_normalized = features_normalized.reshape(features.shape)
    
    # 4. Encode and normalize demographic features
    gender_encoded = scalers['gender_encoder'].transform([gender])[0]
    demo_features = np.array([[age, gender_encoded]])
    demo_normalized = scalers['demographic_scaler'].transform(demo_features)
    
    # 5. Convert to tensors
    x_seq = torch.tensor(features_normalized, dtype=torch.float32).unsqueeze(0)
    x_demo = torch.tensor(demo_normalized, dtype=torch.float32).squeeze(0)
    seq_len = torch.tensor([original_length], dtype=torch.long)
    
    # 6. Make prediction
    model.eval()
    with torch.no_grad():
        outputs = model(x_seq, x_demo, seq_len)
    
    # 7. Convert to original scale
    severity_inverse_mapping = {int(k): int(v) for k, v in mappings['severity_inverse_mapping'].items()}
    
    # Severity (classification)
    severity_class = torch.argmax(outputs['severity_logits'], dim=1).item()
    severity_value = severity_inverse_mapping[severity_class]
    severity_confidence = torch.softmax(outputs['severity_logits'], dim=1)[0, severity_class].item()
    
    # Social Affect (regression)
    social_affect_scaled = outputs['social_affect'].squeeze().item()
    social_affect_value = scalers['social_affect_scaler'].inverse_transform([[social_affect_scaled]])[0, 0]
    
    # RRB (regression)
    rrb_scaled = outputs['rrb'].squeeze().item()
    rrb_value = scalers['rrb_scaler'].inverse_transform([[rrb_scaled]])[0, 0]
    
    # Comparison Score (classification)
    comparison_class = torch.argmax(outputs['comparison_logits'], dim=1).item()
    comparison_value = comparison_class + 1  # 0-9 -> 1-10
    comparison_confidence = torch.softmax(outputs['comparison_logits'], dim=1)[0, comparison_class].item()
    
    predictions = {
        'severity': int(severity_value),
        'severity_confidence': float(severity_confidence),
        'social_affect': float(social_affect_value),
        'rrb': float(rrb_value),
        'comparison_score': int(comparison_value),
        'comparison_confidence': float(comparison_confidence),
        'input_age': age,
        'input_gender': gender,
        'model_type': model_type
    }
    
    # 8. Generate Explainability (if requested)
    if include_explainability:
        explainer = MultiTaskADOSExplainer(
            model=model,
            target_scalers={
                'social_affect_scaler': scalers['social_affect_scaler'],
                'rrb_scaler': scalers['rrb_scaler']
            },
            demographic_scaler=scalers['demographic_scaler'],
            device='cpu'
        )
        
        explanations = explainer.explain_all_tasks(
            x_seq=x_seq.squeeze(0),
            x_demo=x_demo,
            seq_len=seq_len,
            severity_inverse_mapping=severity_inverse_mapping
        )
        
        # Add explainability to predictions
        predictions['explainability'] = {
            'severity': {
                'top_frames': explanations['severity']['top_frames'],
                'top_joints': explanations['severity']['top_joints'],
                'demographic_importance': {
                    'age': float(explanations['severity']['demographic_importance'][0]),
                    'gender': float(explanations['severity']['demographic_importance'][1])
                }
            },
            'social_affect': {
                'top_frames': explanations['social_affect']['top_frames'],
                'top_joints': explanations['social_affect']['top_joints'],
                'demographic_importance': {
                    'age': float(explanations['social_affect']['demographic_importance'][0]),
                    'gender': float(explanations['social_affect']['demographic_importance'][1])
                }
            },
            'rrb': {
                'top_frames': explanations['rrb']['top_frames'],
                'top_joints': explanations['rrb']['top_joints'],
                'demographic_importance': {
                    'age': float(explanations['rrb']['demographic_importance'][0]),
                    'gender': float(explanations['rrb']['demographic_importance'][1])
                }
            },
            'comparison_score': {
                'top_frames': explanations['comparison_score']['top_frames'],
                'top_joints': explanations['comparison_score']['top_joints'],
                'demographic_importance': {
                    'age': float(explanations['comparison_score']['demographic_importance'][0]),
                    'gender': float(explanations['comparison_score']['demographic_importance'][1])
                }
            }
        }
    
    return predictions

# Example usage:
print("✅ Complete inference function with explainability and security features defined")
print("\n🛡️ Security Features:")
print("   - Defensive preprocessing (temporal smoothing + median filtering)")
print("   - Trained with adversarial examples (FGSM/PGD)")
print("   - Robust to noise and perturbations")
print("\nExample usage:")
print("  predictions = make_prediction_on_new_video(")
print("      video_folder_path='/path/to/video/frames',")
print("      age=8.5,")
print("      gender='M',")
print("      model_type='2D',  # or '3D'")
print("      include_explainability=True,")
print("      apply_defensive_preprocessing=True  # 🛡️ Security feature")
print("  )")
print("\nOutput:")
print("  {")
print("      'severity': 2 or 3,")
print("      'severity_confidence': 0.85,")
print("      'social_affect': 8.5,")
print("      'rrb': 3.2,")
print("      'comparison_score': 7 (1-10),")
print("      'comparison_confidence': 0.92,")
print("      'input_age': 8.5,")
print("      'input_gender': 'M',")
print("      'model_type': '2D',")
print("      'explainability': {")
print("          'severity': {")
print("              'top_frames': [45, 67, 23, 89, 12],")
print("              'top_joints': ['left_wrist', 'right_elbow', 'nose', 'left_shoulder', 'neck'],")
print("              'demographic_importance': {'age': 0.23, 'gender': 0.18}")
print("          },")
print("          'social_affect': {...},")
print("          'rrb': {...},")
print("          'comparison_score': {...}")
print("      }")
print("  }")